# 01 Synthetic Data Engineering（可选）

本轮默认任务为 Numeral System（Easy）、Gravity（Medium）、Equation Numeric（Hard）。
以课程公开先验和真实 Train 分布生成小型 Pilot，逐条执行 **Generator → Solver → 独立 Verifier → Leakage Audit → Screening**。
Bit 的完整实现保留为高级扩展，默认调度、数据集及 Preview 均不包含 Bit。

本节可完全跳过，不阻塞 [00 Baseline Diagnosis](00_Baseline_Diagnosis.ipynb)。
后训练 Notebook 的算法和旧数据接口本轮冻结；这里的新版本 Accepted 不会自动合并到旧训练数据。


当前默认基座为 `Qwen/Qwen3-0.6B`，保留官方 Thinking 模板。原 GPU Baseline 与 SFT/DPO/PPO/GRPO 资源 Smoke 已完成；新版低显存实现已通过本地 CPU 检查，尚待 GPU 复测。见 [基准报告](docs/QWEN3_0_6B_BENCHMARK.md) 与 [低显存说明](docs/QWEN3_0_6B_LOW_MEMORY.md)。


## 1. 配置与数据交接

读取 `tasksets/roman_gravity_equation_v1/` 的版本化 Manifest。
`RUN_SYNTHETIC=0` 默认不生成、不自动重审历史 Pilot；如需生成，设置为 `1`，并选择新的 `SYNTHETIC_RUN_ID`。
默认每类只生成 5 条候选用于 CPU 机制检查，可通过 `SYNTHETIC_CANDIDATES_PER_TASK` 调整。
生成只使用 Train 的题面分布；Validation、Holdout 只加载 ID、Prompt 等审计字段，不读取答案。
本节不安装依赖或下载模型，NeMo Data Designer 仍为独立选做分支，本轮不调用。


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import time
import re
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from functools import lru_cache
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
dataset_version = os.environ.get("TASKSET_VERSION", "roman_gravity_equation_v1")
synthetic_run_id = os.environ.get("SYNTHETIC_RUN_ID", "pilot_v1")
if any(not re.fullmatch(r"[A-Za-z0-9_-]+", value) for value in (dataset_version, synthetic_run_id)):
    raise ValueError("非法版本或 Run ID")
output_dir = project_root / "data" / "processed" / "tasksets" / dataset_version
split_dir = output_dir / "splits"
synthetic_dir = output_dir / "synthetic" / synthetic_run_id
synthetic_dir.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get("QWEN3_MODEL_ID", "Qwen/Qwen3-0.6B")
MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1].lower().replace(".", "_")
DEFAULT_MODEL_PATH = Path.cwd() / "models" / MODEL_ID.rsplit("/", 1)[-1]
if not DEFAULT_MODEL_PATH.is_dir():
    DEFAULT_MODEL_PATH = Path.home() / "autodl-tmp/huggingface/models" / MODEL_ID.rsplit("/", 1)[-1]
MODEL_SOURCE = os.environ.get("QWEN3_MODEL_PATH", os.environ.get("LOCAL_MODEL_PATH",
    str(DEFAULT_MODEL_PATH) if DEFAULT_MODEL_PATH.is_dir() else MODEL_ID))
# Catch a stale 4B path before recording it under the 0.6B model ID.
if MODEL_ID == "Qwen/Qwen3-0.6B" and Path(MODEL_SOURCE).is_dir():
    _local_config = json.loads((Path(MODEL_SOURCE) / "config.json").read_text())
    if (_local_config.get("model_type"), _local_config.get("hidden_size"), _local_config.get("num_hidden_layers")) != ("qwen3", 1024, 28):
        raise ValueError("QWEN3_MODEL_PATH 与 Qwen3-0.6B 不匹配；请检查旧模型环境变量")
TASK_ORDER = ("numeral_system", "gravity", "equation_numeric")
TARGET_TASKS = set(TASK_ORDER)
SEED = 42
run_synthetic = os.environ.get("RUN_SYNTHETIC", "0") == "1"
run_data_designer_preview = os.environ.get("RUN_DATA_DESIGNER_PREVIEW", "0") == "1"
pilot_candidates_per_task = int(os.environ.get("SYNTHETIC_CANDIDATES_PER_TASK", "5"))
assert pilot_candidates_per_task > 0
manifest_path = output_dir / "dataset_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("请先运行 00_Baseline_Diagnosis.ipynb 保存划分与 Manifest")
dataset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert set(dataset_manifest["tasks"]) == TARGET_TASKS, "Manifest 任务集合不匹配"
audit_frames = []
for name in ("train", "validation", "holdout_test"):
    path = project_root / dataset_manifest["files"][name]
    if hashlib.sha256(path.read_bytes()).hexdigest() != dataset_manifest["artifacts"][name]["sha256"]:
        raise ValueError(f"{name} 与 Manifest 不一致，请重新运行 00")
    columns = ["source_row", "source_dataset", "id", "prompt", "answer", "task_type", "split"]
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, usecols=columns)
    assert frame["split"].eq(name).all()
    assert len(frame) == dataset_manifest["artifacts"][name]["rows"]
    audit_frames.append(frame)
split_df = pd.concat(audit_frames, ignore_index=True)
train_df = split_df[split_df["split"] == "train"].copy()
print(split_df.groupby("split").size().to_dict())

In [ ]:
def prompt_hash(text: object) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def normalize_prompt(text: object) -> str:
    # 保留数字、二进制和运算符；掩去数值只用于结构相似性审计。
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(text)).strip()).casefold()


def structure_fingerprint(text, task_type):
    text = normalize_prompt(text)
    if task_type == "numeral_system": text = re.sub(r"\b[ivxlcdm]+\b", "<roman>", text)
    text = re.sub(r"\d{2}" if task_type == "equation_numeric" else r"[-+]?\d+(?:\.\d+)?", "<num>", text)
    return prompt_hash(text)

# Historical advanced Bit audit, outside the default taskset.
def advanced_bit_structure_fingerprint(text: object, task_type: str) -> str:
    normalized = normalize_prompt(text)
    normalized = re.sub(r"0b[01]+", "<bit>", normalized)
    if task_type == "equation_numeric":
        normalized = re.sub(r"\d{2}", "<2d>", normalized)
    else:
        normalized = re.sub(r"[-+]?\d+(?:\.\d+)?", "<num>", normalized)
    return prompt_hash(normalized)

In [ ]:
def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=re.I)
    return text.strip()

OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v3"

OFFICIAL_NUMERIC_REL_TOL = 1e-2

OFFICIAL_NUMERIC_ABS_TOL = 1e-5

def official_numeric_match(predicted: object, expected: object) -> bool:
    """NVIDIA metric-compatible numeric comparison for answers and Verifiers."""
    try:
        predicted_value = float(normalize_answer_text(predicted))
        expected_value = float(normalize_answer_text(expected))
    except (TypeError, ValueError):
        return False
    if not (math.isfinite(predicted_value) and math.isfinite(expected_value)):
        return False
    return math.isclose(
        predicted_value,
        expected_value,
        rel_tol=OFFICIAL_NUMERIC_REL_TOL,
        abs_tol=OFFICIAL_NUMERIC_ABS_TOL,
    )

def numeric_or_text_match(predicted: object, expected: object) -> bool:
    """Use the official tolerance for numeric strings and exact matching otherwise."""
    predicted_text, expected_text = normalize_answer_text(predicted), normalize_answer_text(expected)
    if official_numeric_match(predicted_text, expected_text):
        return True
    return predicted_text == expected_text

## 2. 课程先验与 Train 分布

### 2.1 固定任务先验


In [ ]:
GENERATOR_VERSION = "nemotron-synthetic-taskset-v4"

gravity_prior = {
    "formula": "distance = 0.5 * g * t^2",
    "units": {"t": "seconds", "distance": "meters", "g": "m/s^2"},
    "sampling_source": "train_split_empirical_profile",
    "answer_precision": "observed_decimal_places_with_official_tolerance",
}
BIT_PRIOR = {
    "width": 8,
    "operations": ["SHL", "SHR", "ROTL", "ROTR", "AND", "OR", "XOR", "NOT"],
    "expression_language": "x is the input byte; unary and binary trees have maximum depth 2",
    "mask": 255,
}

### 2.1 Numeral System：标准先验、Train 采样与独立验证

标准 Roman 范围为 1–3999，真实 Train 的数值范围和示例数从题面提取。Generator 只构造候选；Solver 用贪心分解生成轨迹，Verifier 用规范语法和减法求值独立验证。Query 不重复当前题内示例，跨集合有限映射暴露仍须单独报告。


In [ ]:
ROMAN_GRAMMAR = re.compile(r"M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})")
ROMAN_VALUES = {"I":1, "V":5, "X":10, "L":50, "C":100, "D":500, "M":1000}

def arabic_to_roman(number):
    if not isinstance(number, int) or isinstance(number, bool) or not 1 <= number <= 3999: raise ValueError("标准范围为 1..3999")
    thousands = ("", "M", "MM", "MMM")
    hundreds = ("", "C", "CC", "CCC", "CD", "D", "DC", "DCC", "DCCC", "CM")
    tens = ("", "X", "XX", "XXX", "XL", "L", "LX", "LXX", "LXXX", "XC")
    ones = ("", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX")
    return thousands[number//1000] + hundreds[number//100%10] + tens[number//10%10] + ones[number%10]

def canonical_roman_to_int(roman):
    if not isinstance(roman, str) or not roman or not ROMAN_GRAMMAR.fullmatch(roman): raise ValueError("非规范罗马数字")
    values = [ROMAN_VALUES[char] for char in roman]
    result = sum(-value if index+1 < len(values) and value < values[index+1] else value for index, value in enumerate(values))
    if not 1 <= result <= 3999: raise ValueError("超出标准范围")
    return result

def verify_roman_number(number, roman):
    if not re.fullmatch(r"[0-9]+", str(number)): return False
    try: return canonical_roman_to_int(roman) == int(number)
    except (TypeError, ValueError, KeyError): return False

def parse_numeral_prompt(prompt):
    pairs = [(int(a), b) for a,b in re.findall(r"(?m)^(\d+)\s*->\s*([A-Z]+)\s*$", prompt)]
    mapping_lines = [line for line in prompt.splitlines() if "->" in line]
    query = re.search(r"Now, write the number (\d+) in the Wonderland numeral system\.\s*$", prompt)
    if not query or not pairs or len(mapping_lines) != len(pairs): raise ValueError("Numeral 题面解析失败")
    return pairs, int(query.group(1))



In [ ]:
# Only Train prompt statistics determine sampling; held-out answers are used only by the mapping audit.
_train_roman = [parse_numeral_prompt(prompt) for prompt in train_df[train_df.task_type == "numeral_system"].prompt]
roman_number_pool = sorted({value for pairs, query in _train_roman for value in [query, *[number for number, _ in pairs]]})
roman_example_counts = [len(pairs) for pairs, _ in _train_roman]
assert roman_number_pool and max(roman_example_counts) < len(roman_number_pool)

def generate_numeral_sample(rng):
    count = int(rng.choice(roman_example_counts))
    values = rng.choice(roman_number_pool, size=count + 1, replace=False).tolist()
    examples, query = values[:-1], values[-1]
    prompt = "In the Wonderland numeral system, numbers are represented as follows:\n"
    prompt += "\n".join(f"{number} -> {arabic_to_roman(number)}" for number in examples)
    prompt += f"\nNow, write the number {query} in the Wonderland numeral system."
    return {"task_type": "numeral_system", "source": "deterministic_prior_generator",
            "generator_version": GENERATOR_VERSION, "hidden_rule": {"system": "standard_roman", "range": [min(roman_number_pool), max(roman_number_pool)]},
            "prompt": prompt, "answer": "", "trace": {}, "difficulty": "easy", "deduce_or_guess": "deduce",
            "generation_status": "generated", "generation_error": None, "generation_seed": SEED}

def solve_numeral_prompt(prompt):
    examples, query = parse_numeral_prompt(prompt)
    if not all(verify_roman_number(number, roman) for number, roman in examples):
        raise ValueError("示例不满足公开标准 Roman 先验")
    remaining, components = query, []
    # Solver uses greedy subtraction; example encoder uses place-value tables.
    for value, token in ((1000,"M"),(900,"CM"),(500,"D"),(400,"CD"),(100,"C"),(90,"XC"),(50,"L"),(40,"XL"),(10,"X"),(9,"IX"),(5,"V"),(4,"IV"),(1,"I")):
        while remaining >= value:
            components.append({"arabic": value, "roman": token}); remaining -= value
    return {"status": "solved", "answer": "".join(part["roman"] for part in components),
            "trace": {"query_number": query, "components": components}}

def verify_numeral_sample(sample):
    # Independent decoder validates grammar and subtractive value; no Solver call.
    try:
        examples, query = parse_numeral_prompt(str(sample.get("prompt", "")))
        examples_ok = all(verify_roman_number(number, roman) for number, roman in examples)
        answer = str(sample.get("answer", "")); trace = sample.get("trace", {})
        parts = trace.get("components", [])
        trace_ok = bool(parts) and trace.get("query_number") == query
        trace_ok = trace_ok and all(verify_roman_number(part["arabic"], part["roman"]) for part in parts)
        trace_ok = trace_ok and sum(part["arabic"] for part in parts) == query
        trace_ok = trace_ok and "".join(part["roman"] for part in parts) == answer
        return {"answer_ok": examples_ok and verify_roman_number(query, answer), "trace_ok": examples_ok and bool(trace_ok),
                "identifiability": {"answer_unique": examples_ok and 1 <= query <= 3999,
                                    "scope": "public standard Roman prior, not arbitrary hidden numeral systems"},
                "difficulty_ok": sample.get("difficulty") == "easy", "examples_ok": examples_ok}
    except (ValueError, TypeError, KeyError, AttributeError):
        return {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False}


### 2.2 Equation Numeric Prior：24 条规则与全局约束

把操作数解析、算术操作、输出格式和跨运算符约束放在独立单元，后面的 Generator、Solver 与 Verifier 共享规格但不共享证明路径。

In [ ]:
# Numeric Equation prior extracted from the reference solver's DSL and Numeric24 rule table.
# A rule is (operand parse, arithmetic operation, output format); it is not merely an
# operator -> answer lookup. Global assignments additionally enforce one family and one
# non-join semantic group per prompt, while join rules may coexist.
NUMERIC24_RULES = [
    {"name": "sub", "parse": "AB_CD", "operation": "sub_lr", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff", "parse": "AB_CD", "operation": "abs_sub", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff_prefix", "parse": "AB_CD", "operation": "abs_sub", "format": "sign_prefix_raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "join", "parse": "AB_CD", "operation": "concat_lr", "format": "raw", "family": "join", "group": "join_like"},
    {"name": "join_swap", "parse": "AB_CD", "operation": "concat_rl", "format": "raw", "family": "join", "group": "join_like"},
    {"name": "add", "parse": "AB_CD", "operation": "add", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "add_p1", "parse": "AB_CD", "operation": "add_p1", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "add_m1", "parse": "AB_CD", "operation": "add_m1", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "mul", "parse": "AB_CD", "operation": "mul", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "mul_p1", "parse": "AB_CD", "operation": "mul_p1", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "mul_m1", "parse": "AB_CD", "operation": "mul_m1", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "maxmodmin", "parse": "AB_CD", "operation": "max_mod_min", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "flip_add", "parse": "BA_DC", "operation": "add", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_add_p1", "parse": "BA_DC", "operation": "add_p1", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_add_m1", "parse": "BA_DC", "operation": "add_m1", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_mul", "parse": "BA_DC", "operation": "mul", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_p1", "parse": "BA_DC", "operation": "mul_p1", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_m1", "parse": "BA_DC", "operation": "mul_m1", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_absdiff", "parse": "BA_DC", "operation": "abs_sub", "format": "rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_maxmodmin", "parse": "BA_DC", "operation": "max_mod_min", "format": "rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_prefix", "parse": "BA_DC", "operation": "sub_lr", "format": "sign_prefix_rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_suffix", "parse": "BA_DC", "operation": "sub_rl", "format": "rev_or_neg_suffix", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_prefix", "parse": "BA_DC", "operation": "abs_sub", "format": "always_prefix_rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_suffix", "parse": "BA_DC", "operation": "abs_sub", "format": "always_suffix_rev", "family": "flip", "group": "sub_or_mod_like"},
]
NUMERIC24_RULE_BY_NAME = {rule["name"]: rule for rule in NUMERIC24_RULES}
NUMERIC24_FORMULAS = {
    "sub": "AB - CD, raw", "absdiff": "|AB - CD|, raw", "absdiff_prefix": "|AB - CD| with operator prefix",
    "join": "AB concatenated with CD", "join_swap": "CD concatenated with AB",
    "add": "AB + CD", "add_p1": "AB + CD + 1", "add_m1": "AB + CD - 1",
    "mul": "AB * CD", "mul_p1": "AB * CD + 1", "mul_m1": "AB * CD - 1",
    "maxmodmin": "max(AB, CD) mod min(AB, CD)",
    "flip_add": "reverse(BA + DC)", "flip_add_p1": "reverse(BA + DC + 1)", "flip_add_m1": "reverse(BA + DC - 1)",
    "flip_mul": "reverse(BA * DC)", "flip_mul_p1": "reverse(BA * DC + 1)", "flip_mul_m1": "reverse(BA * DC - 1)",
    "flip_absdiff": "reverse(|BA - DC|)", "flip_maxmodmin": "reverse(max(BA, DC) mod min(BA, DC))",
    "flip_sub_prefix": "reverse(BA - DC), preserving a negative sign as an operator prefix",
    "flip_sub_suffix": "reverse(DC - BA), appending the operator for a negative result",
    "flip_absdiff_prefix": "operator-prefixed reverse(|BA - DC|)",
    "flip_absdiff_suffix": "reverse(|BA - DC|) followed by the operator",
}
EQUATION_NUMERIC_PRIOR = {
    "operand_pattern": "AB<operator>CD with two decimal digits on each side",
    "parses": {"AB_CD": "left=AB, right=CD", "BA_DC": "left=reverse(AB), right=reverse(CD)"},
    "operations": ["concatenate", "add", "add±1", "subtract", "reverse subtract", "absolute difference", "multiply", "multiply±1", "max mod min"],
    "output_formats": ["raw", "reverse digits", "operator-prefixed reverse", "two-digit padding", "operator suffix"],
    "rules": [{"name": name, "formula": NUMERIC24_FORMULAS[name]} for name in NUMERIC24_RULE_BY_NAME],
    "global_constraints": [
        "All non-join rules in one prompt use either normal or flipped operand family.",
        "At most one non-join rule from each semantic group: sub_or_mod_like, add_like, mul_like.",
        "Join and reverse-join are the join_like exception and can coexist with the other groups.",
        "A query operator absent from examples is Guess; it is admitted only when all valid assignments give one query output.",
    ],
}


### 2.3 Gravity 真实 Train 分布审计函数

先定义时间、距离、重力估计和残差的统计函数。

In [ ]:
# The synthetic distribution is fitted from the Train split only. Validation and
# Holdout rows never enter these arrays or the generator seed.
GRAVITY_PAIR_RE = re.compile(r"For t = ([-+]?\d+(?:\.\d+)?)s, distance = ([-+]?\d+(?:\.\d+)?) m")


def _gravity_quantiles(values: list[float]) -> dict[str, float]:
    values = [float(value) for value in values]
    quantile_levels = {"q01": 0.01, "q05": 0.05, "q25": 0.25, "median": 0.50, "q75": 0.75, "q95": 0.95, "q99": 0.99}
    array = np.asarray(values, dtype=float)
    return {
        "count": int(array.size),
        "min": float(array.min()),
        **{name: float(np.quantile(array, level)) for name, level in quantile_levels.items()},
        "max": float(array.max()),
        "mean": float(array.mean()),
        "std": float(array.std()),
    }


def _extract_gravity_profile(frame: pd.DataFrame) -> dict[str, object]:
    records = []
    for row in frame.itertuples(index=False):
        pairs = [(float(t), float(distance)) for t, distance in GRAVITY_PAIR_RE.findall(str(row.prompt))]
        if len(pairs) < 2:
            continue
        times = np.asarray([t for t, _ in pairs], dtype=float)
        distances = np.asarray([distance for _, distance in pairs], dtype=float)
        g_i = 2.0 * distances / np.square(times)
        g_ols = 2.0 * float(np.sum(np.square(times) * distances)) / float(np.sum(np.power(times, 4)))
        g_median = float(np.median(g_i))
        residuals = distances - 0.5 * g_ols * np.square(times)
        decimals = [len(str(distance).split(".", 1)[1]) if "." in str(distance) else 0 for _, distance in pairs]
        records.append({
            "source_row": int(getattr(row, "source_row", -1)),
            "id": str(getattr(row, "id", "")),
            "example_count": len(pairs),
            "times": times.tolist(),
            "distances": distances.tolist(),
            "g_i": g_i.tolist(),
            "g_ols": g_ols,
            "g_median": g_median,
            "residuals": residuals.tolist(),
            "distance_decimal_places": decimals,
        })
    if not records:
        raise ValueError("No parseable Gravity rows in Train split")
    return {
        "source_split": "train",
        "row_count": len(records),
        "g_ols_values": [record["g_ols"] for record in records],
        "g_median_values": [record["g_median"] for record in records],
        "t_values": [t for record in records for t in record["times"]],
        "d_values": [d for record in records for d in record["distances"]],
        "example_counts": [record["example_count"] for record in records],
        "residual_values": [residual for record in records for residual in record["residuals"]],
        "distance_decimal_places": [places for record in records for places in record["distance_decimal_places"]],
        "audit": {
            "example_count_distribution": {str(k): int(v) for k, v in Counter(record["example_count"] for record in records).items()},
            "t": _gravity_quantiles([t for record in records for t in record["times"]]),
            "distance": _gravity_quantiles([d for record in records for d in record["distances"]]),
            "g_i": _gravity_quantiles([g for record in records for g in record["g_i"]]),
            "g_ols": _gravity_quantiles([record["g_ols"] for record in records]),
            "g_median": _gravity_quantiles([record["g_median"] for record in records]),
            "residual": _gravity_quantiles([residual for record in records for residual in record["residuals"]]),
            "distance_decimal_places": {str(k): int(v) for k, v in Counter(place for record in records for place in record["distance_decimal_places"]).items()},
        },
    }


### 2.4 执行 Gravity 分布审计并保存对比图

这一格只使用 Train 划分拟合合成分布，并保存审计 JSON 和新旧采样分布图。

In [ ]:
train_gravity_frame = split_df[(split_df["split"] == "train") & (split_df["task_type"] == "gravity")].copy()
gravity_train_profile = _extract_gravity_profile(train_gravity_frame)
gravity_prior["observed_train_profile"] = gravity_train_profile["audit"]
gravity_audit_path = synthetic_dir / "gravity_distribution_audit.json"
gravity_audit_path.write_text(json.dumps({
    "schema_version": "nemotron-course-gravity-audit/v1",
    "source_split": "train",
    "source_rows": gravity_train_profile["row_count"],
    "formula": "g_i=2*d_i/t_i^2; g_ols=2*sum(t_i^2*d_i)/sum(t_i^4)",
    "statistics": gravity_train_profile["audit"],
    "generator_sampling": {
        "g": "empirical Train g_ols values",
        "t": "empirical Train example times",
        "example_count": "empirical Train per-question counts",
        "distance_precision": "empirical Train decimal-place frequencies",
        "noise": "no additional noise; distance is rounded at sampled observed precision",
    },
}, ensure_ascii=False, indent=2), encoding="utf-8")

legacy_g_grid = np.arange(9.50, 19.50 + 0.005, 0.01)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(gravity_train_profile["g_ols_values"], bins=30, alpha=0.75, label="Train g OLS", color="#4C78A8")
axes[0, 0].hist(legacy_g_grid, bins=30, alpha=0.35, label="legacy uniform grid", color="#F58518")
axes[0, 0].set_title("Gravity g distribution: Train vs legacy")
axes[0, 0].set_xlabel("estimated g")
axes[0, 0].set_ylabel("count")
axes[0, 0].legend()
axes[0, 1].hist(gravity_train_profile["t_values"], bins=30, color="#54A24B")
axes[0, 1].set_title("Train example time distribution")
axes[0, 1].set_xlabel("t (s)")
axes[0, 1].set_ylabel("count")
axes[1, 0].hist(gravity_train_profile["d_values"], bins=30, color="#B279A2")
axes[1, 0].set_title("Train example distance distribution")
axes[1, 0].set_xlabel("distance (m)")
axes[1, 0].set_ylabel("count")
axes[1, 1].hist(gravity_train_profile["residual_values"], bins=30, color="#E45756")
axes[1, 1].set_title("OLS fit residual distribution")
axes[1, 1].set_xlabel("distance residual (m)")
axes[1, 1].set_ylabel("count")
fig.tight_layout()
gravity_comparison_plot = synthetic_dir / "gravity_distribution_comparison.png"
fig.savefig(gravity_comparison_plot, dpi=150)
plt.show()


### 2.5 真实 Equation Operator 清单

从 Train Prompt 提取实际出现的运算符，避免合成池静默遗漏真实符号。

In [ ]:
def _extract_train_numeric_operators(frame: pd.DataFrame) -> tuple[str, ...]:
    operators = set()
    for prompt in frame.loc[frame["task_type"] == "equation_numeric", "prompt"]:
        for line in str(prompt).splitlines():
            example = re.match(r"^\s*\d{2}([^\d\s])\d{2}\s*=", line)
            query = re.search(r"Now, determine the result for:\s*\d{2}([^\d\s])\d{2}", line)
            if example:
                operators.add(example.group(1))
            if query:
                operators.add(query.group(1))
    return tuple(sorted(operators))


train_numeric_operator_pool = _extract_train_numeric_operators(split_df[split_df["split"] == "train"])
numeric_operator_inventory = {
    "train_observed": list(train_numeric_operator_pool),
    "count": len(train_numeric_operator_pool),
    "unsupported_by_prior": [],
}
(synthetic_dir / "equation_operator_inventory.json").write_text(json.dumps(numeric_operator_inventory, ensure_ascii=False, indent=2), encoding="utf-8")


print(json.dumps({"gravity_prior": gravity_prior, "bit_prior": BIT_PRIOR, "equation_numeric_rules": len(NUMERIC24_RULES), "pilot_candidates_per_task": pilot_candidates_per_task}, ensure_ascii=False, indent=2))


### 2.6 Gravity：渲染与题面解析

In [ ]:
# Gravity: one-shot candidate generator, prompt solver, and independent verifier.
# The verifier has its own floating-point fit and never calls generator helpers.
def _gravity_generator_render(distance_t: float | Decimal, g_value: float | Decimal, decimal_places: int = 2) -> str:
    t = Decimal(str(distance_t))
    g = Decimal(str(g_value))
    quantizer = Decimal(1).scaleb(-int(decimal_places))
    return format((Decimal("0.5") * g * t * t).quantize(quantizer), "f")


# Kept as a named generator-side helper for traceability; verifier code below does not call it.
def _gravity_render(distance_t: float | Decimal, g_value: float | Decimal) -> str:
    return _gravity_generator_render(distance_t, g_value, decimal_places=2)


def _gravity_prompt_parts(prompt: str) -> tuple[list[tuple[str, str]], str | None]:
    examples = [(t, d) for t, d in re.findall(r"For t = ([-+]?\d+(?:\.\d+)?)s, distance = ([-+]?\d+(?:\.\d+)?) m", str(prompt))]
    query_match = re.search(r"falling distance for t = ([-+]?\d+(?:\.\d+)?)s", str(prompt))
    return examples, query_match.group(1) if query_match else None


### 2.7 Gravity：独立 Solver 与重力估计

In [ ]:
def solve_gravity_prompt(prompt: str) -> dict[str, object]:
    """Solver: infer g from the public examples using OLS, then answer the query."""
    examples, query_t = _gravity_prompt_parts(prompt)
    if len(examples) < 2 or query_t is None:
        return {"status": "invalid_prompt", "answer": None, "inferred_g": None, "trace": {}}
    times = np.asarray([float(t) for t, _ in examples], dtype=float)
    distances = np.asarray([float(d) for _, d in examples], dtype=float)
    denominator = float(np.sum(np.power(times, 4)))
    if denominator <= 0:
        return {"status": "invalid_prompt", "answer": None, "inferred_g": None, "trace": {}}
    g_ols = 2.0 * float(np.sum(np.square(times) * distances)) / denominator
    query_raw = 0.5 * g_ols * float(query_t) ** 2
    answer = f"{query_raw:.2f}"
    trace = {
        "solver": "ols_from_public_examples",
        "g_estimates": [2.0 * d / (t * t) for t, d in zip(times.tolist(), distances.tolist())],
        "g_ols": g_ols,
        "query_t": str(query_t),
        "query_raw": query_raw,
    }
    return {"status": "solved", "answer": answer, "inferred_g": g_ols, "trace": trace}


### 2.8 Gravity：Verifier 拟合、答案与可识别性

In [ ]:
def _gravity_verifier_fit(examples: list[tuple[str, str]]) -> dict[str, object]:
    """Independent verifier-side parameter fit; no generator or Solver call."""
    try:
        times = np.asarray([float(t) for t, _ in examples], dtype=float)
        distances = np.asarray([float(d) for _, d in examples], dtype=float)
    except (TypeError, ValueError):
        return {"valid": False, "g_ols": None, "g_median": None, "residuals": []}
    if len(times) < 2 or np.any(times <= 0) or not np.all(np.isfinite(distances)):
        return {"valid": False, "g_ols": None, "g_median": None, "residuals": []}
    denominator = float(np.sum(np.power(times, 4)))
    if denominator <= 0:
        return {"valid": False, "g_ols": None, "g_median": None, "residuals": []}
    per_example = 2.0 * distances / np.square(times)
    g_ols = 2.0 * float(np.sum(np.square(times) * distances)) / denominator
    g_median = float(np.median(per_example))
    residuals = (distances - 0.5 * g_ols * np.square(times)).tolist()
    return {
        "valid": True,
        "g_ols": g_ols,
        "g_median": g_median,
        "g_estimates": per_example.tolist(),
        "residuals": residuals,
        "times": times.tolist(),
        "distances": distances.tolist(),
    }


def _gravity_verifier_distance(time_value: float, g_value: float) -> float:
    # Separate implementation from Decimal-based generator rendering.
    return 0.5 * float(g_value) * float(time_value) * float(time_value)


def gravity_identifiability(prompt: str) -> dict[str, object]:
    examples, query_t = _gravity_prompt_parts(prompt)
    fit = _gravity_verifier_fit(examples)
    if not fit["valid"] or query_t is None:
        return {"candidate_g_count": 0, "query_output_count": 0, "query_outputs": [], "answer_unique": False}
    candidates = [float(fit["g_ols"]), float(fit["g_median"])]
    query_outputs = [_gravity_verifier_distance(float(query_t), g) for g in candidates]
    unique = all(official_numeric_match(query_outputs[0], value) for value in query_outputs[1:])
    return {
        "candidate_g_count": len(candidates),
        "candidate_g_values": candidates,
        "query_output_count": 1 if unique else len(query_outputs),
        "query_outputs": [query_outputs[0]] if unique else query_outputs,
        "answer_unique": unique,
    }


### 2.9 Gravity：单次候选 Generator

In [ ]:
def generate_gravity_sample(rng: np.random.Generator) -> dict[str, object]:
    """Generator: one attempt from the empirical Train-only Gravity profile."""
    profile = gravity_train_profile
    example_count = int(rng.choice(np.asarray(profile["example_counts"], dtype=int)))
    g_value = float(rng.choice(np.asarray(profile["g_ols_values"], dtype=float)))
    times = rng.choice(np.asarray(profile["t_values"], dtype=float), size=example_count + 1, replace=False)
    decimal_places = rng.choice(np.asarray(profile["distance_decimal_places"], dtype=int), size=example_count, replace=True)
    example_pairs = [
        (f"{float(t):g}", _gravity_generator_render(float(t), g_value, int(places)))
        for t, places in zip(times[:-1], decimal_places)
    ]
    query_t = f"{float(times[-1]):g}"
    prompt = (
        "The course prior defines the physical relation distance = 0.5*g*t^2, with t in seconds and distance in meters.\n"
        + "\n".join(f"For t = {t}s, distance = {distance} m" for t, distance in example_pairs)
        + f"\nNow, calculate the falling distance for t = {query_t}s."
    )
    return {
        "task_type": "gravity",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"g": repr(g_value), "formula": gravity_prior["formula"], "profile_source": "train"},
        "prompt": prompt,
        "answer": "",
        "trace": {},
        "difficulty": f"examples_{example_count}",
        "deduce_or_guess": "deduce",
        "generation_status": "generated",
        "generation_error": None,
    }


### 2.10 Gravity：独立样本 Verifier

In [ ]:
def verify_gravity_sample(sample: dict[str, object]) -> dict[str, object]:
    """Verifier: independently fit the public examples using official isclose."""
    examples, query_t = _gravity_prompt_parts(str(sample.get("prompt", "")))
    fit = _gravity_verifier_fit(examples)
    answer = str(sample.get("answer", ""))
    example_checks = []
    if fit["valid"]:
        for time_value, expected_distance in examples:
            predicted_distance = _gravity_verifier_distance(float(time_value), float(fit["g_ols"]))
            example_checks.append(official_numeric_match(predicted_distance, expected_distance))
    answer_prediction = None
    if fit["valid"] and query_t is not None:
        answer_prediction = _gravity_verifier_distance(float(query_t), float(fit["g_ols"]))
    answer_ok = bool(fit["valid"] and example_checks and all(example_checks) and answer_prediction is not None and official_numeric_match(answer, answer_prediction))
    trace = sample.get("trace", {})
    trace_ok = isinstance(trace, dict) and trace.get("solver") == "ols_from_public_examples"
    if trace_ok:
        trace_ok = trace.get("query_t") == query_t and official_numeric_match(trace.get("g_ols"), fit.get("g_ols"))
        trace_ok = trace_ok and len(trace.get("g_estimates", [])) == len(examples)
    ident = gravity_identifiability(str(sample.get("prompt", "")))
    return {
        "answer_ok": answer_ok,
        "trace_ok": bool(trace_ok),
        "identifiability": ident,
        "example_checks": example_checks,
        "query_prediction": answer_prediction,
        "difficulty_ok": str(sample.get("difficulty", "")).startswith("examples_"),
        "solver_status": "independent_ols_verifier",
    }


### 2.11 高级扩展：Bit DSL 与 Generator 求值路径（不进入默认任务）

保留课程规定的 8-bit DSL，并递归处理二元表达式左右子树。

In [ ]:
# Bit Manipulation: one-shot generator plus independent solver/verifier paths.
MASK8 = 0xFF
BIT_UNARY_OPS = ("NOT",)
BIT_SHIFT_OPS = ("SHL", "SHR", "ROTL", "ROTR")
BIT_BINARY_OPS = ("AND", "OR", "XOR")


# Generator-side implementation used only to materialize examples from a hidden tree.
def compute_bit_tree(node: tuple) -> int:
    kind = node[0]
    if kind == "lit":
        return int(node[1]) & MASK8
    if kind == "var":
        raise ValueError("uninstantiated_variable")
    if kind in BIT_SHIFT_OPS:
        value, shift = compute_bit_tree(node[1]), int(node[2]) % 8
        if kind == "SHL": return (value << shift) & MASK8
        if kind == "SHR": return (value >> shift) & MASK8
        if kind == "ROTL": return ((value << shift) | (value >> (8 - shift))) & MASK8 if shift else value
        return ((value >> shift) | (value << (8 - shift))) & MASK8 if shift else value
    if kind == "NOT":
        return (~compute_bit_tree(node[1])) & MASK8
    left, right = compute_bit_tree(node[1]), compute_bit_tree(node[2])
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def render_bit_tree(node: tuple) -> str:
    kind = node[0]
    if kind == "var": return "x"
    if kind == "lit": return f"0b{int(node[1]) & MASK8:08b}"
    if kind in BIT_SHIFT_OPS: return f"{kind}({render_bit_tree(node[1])}, {int(node[2])})"
    if kind == "NOT": return f"NOT({render_bit_tree(node[1])})"
    return f"{kind}({render_bit_tree(node[1])}, {render_bit_tree(node[2])})"


def instantiate_bit_tree(tree: tuple, value: int) -> tuple:
    """Instantiate every variable recursively, including both binary subtrees."""
    if tree[0] == "var":
        return ("lit", int(value) & MASK8)
    if tree[0] == "lit":
        return ("lit", int(tree[1]) & MASK8)
    if tree[0] in BIT_UNARY_OPS:
        return (tree[0], instantiate_bit_tree(tree[1], value))
    if tree[0] in BIT_SHIFT_OPS:
        return (tree[0], instantiate_bit_tree(tree[1], value), int(tree[2]) % 8)
    return (tree[0], instantiate_bit_tree(tree[1], value), instantiate_bit_tree(tree[2], value))


def random_bit_tree(rng: np.random.Generator, depth: int = 2) -> tuple:
    if depth <= 0:
        return ("var",)
    kind = str(rng.choice(BIT_SHIFT_OPS + BIT_BINARY_OPS + BIT_UNARY_OPS))
    left = random_bit_tree(rng, depth - 1)
    if kind in BIT_UNARY_OPS:
        return (kind, left)
    if kind in BIT_SHIFT_OPS:
        return (kind, left, int(rng.integers(0, 8)))
    return (kind, left, random_bit_tree(rng, depth - 1))


def _bit_tree_value(tree: tuple, value: int) -> int:
    return compute_bit_tree(instantiate_bit_tree(tree, value))


### 2.12 Bit Solver：有限候选表达式枚举

In [ ]:
def _bit_candidate_trees(max_depth: int = 2) -> tuple[tuple, ...]:
    by_depth = {0: (("var",),)}
    for depth in range(1, max_depth + 1):
        smaller = tuple(tree for d in range(depth) for tree in by_depth[d])
        candidates = []
        candidates.extend(("NOT", child) for child in smaller)
        candidates.extend((op, child, shift) for op in BIT_SHIFT_OPS for child in smaller for shift in range(8))
        candidates.extend((op, left, right) for op in BIT_BINARY_OPS for left in smaller for right in smaller)
        unique = {render_bit_tree(tree): tree for tree in candidates}
        by_depth[depth] = tuple(unique.values())
    all_trees = {render_bit_tree(tree): tree for depth in by_depth.values() for tree in depth}
    return tuple(all_trees.values())


# Solver-side evaluator is separate from the Generator's instantiate/compute path.
def _bit_solver_value(node: tuple, value: int) -> int:
    kind = node[0]
    if kind == "var": return int(value) & MASK8
    if kind == "lit": return int(node[1]) & MASK8
    if kind in BIT_SHIFT_OPS:
        child, shift = _bit_solver_value(node[1], value), int(node[2]) % 8
        if kind == "SHL": return (child << shift) & MASK8
        if kind == "SHR": return (child >> shift) & MASK8
        if kind == "ROTL": return ((child << shift) | (child >> (8 - shift))) & MASK8 if shift else child
        return ((child >> shift) | (child << (8 - shift))) & MASK8 if shift else child
    if kind == "NOT": return (~_bit_solver_value(node[1], value)) & MASK8
    left, right = _bit_solver_value(node[1], value), _bit_solver_value(node[2], value)
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def _bit_solver_tree_value(tree: tuple, value: int) -> int:
    return _bit_solver_value(tree, value)


def identify_bit_expression(example_pairs: list[tuple[int, int]], query_value: int, max_depth: int = 2) -> dict[str, object]:
    matching = []
    for tree in _bit_candidate_trees(max_depth):
        if all(_bit_solver_tree_value(tree, value) == output for value, output in example_pairs):
            matching.append(tree)
    semantic_groups = {}
    for tree in matching:
        signature = tuple(_bit_solver_tree_value(tree, value) for value in range(256))
        semantic_groups.setdefault(signature, tree)
    query_outputs = sorted({_bit_solver_tree_value(tree, query_value) for tree in semantic_groups.values()})
    selected_tree = next(iter(semantic_groups.values()), None)
    return {
        "raw_candidate_count": len(matching),
        "semantic_candidate_count": len(semantic_groups),
        "query_output_count": len(query_outputs),
        "query_outputs": [f"{value:08b}" for value in query_outputs],
        "expression_unique": len(matching) == 1,
        "answer_unique": len(query_outputs) == 1,
        "selected_expression": render_bit_tree(selected_tree) if selected_tree else None,
    }


### 2.13 Bit 表达式解析与 Query 求值

In [ ]:
def _parse_bit_expression(text: str, position: int = 0) -> tuple[tuple, int]:
    while position < len(text) and text[position].isspace(): position += 1
    if text.startswith("x", position): return ("var",), position + 1
    literal = re.match(r"0b([01]{8})", text[position:])
    if literal: return ("lit", int(literal.group(1), 2)), position + literal.end()
    operation = re.match(r"(NOT|SHL|SHR|ROTL|ROTR|AND|OR|XOR)\(", text[position:])
    if operation is None: raise ValueError("invalid_bit_expression")
    op, pos = operation.group(1), position + operation.end()
    left, pos = _parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if op == "NOT":
        if pos >= len(text) or text[pos] != ")": raise ValueError("invalid_not")
        return (op, left), pos + 1
    if pos >= len(text) or text[pos] != ",": raise ValueError("missing_comma")
    pos += 1
    if op in BIT_SHIFT_OPS:
        match = re.match(r"\s*([0-7])\s*\)", text[pos:])
        if match is None: raise ValueError("invalid_shift")
        return (op, left, int(match.group(1))), pos + match.end()
    right, pos = _parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if pos >= len(text) or text[pos] != ")": raise ValueError("missing_close")
    return (op, left, right), pos + 1


def _evaluate_bit_expression(text: str, value: int) -> int:
    tree, position = _parse_bit_expression(text)
    if text[position:].strip(): raise ValueError("trailing_expression")
    return _bit_solver_tree_value(tree, value)


def solve_bit_prompt(prompt: str) -> dict[str, object]:
    pairs = [(int(value, 2), int(output, 2)) for value, output in re.findall(r"Input: 0b([01]{8}) -> Output: 0b([01]{8})", str(prompt))]
    query_match = re.search(r"output for input: 0b([01]{8})", str(prompt))
    if not query_match or not pairs:
        return {"status": "invalid_prompt", "answer": None, "trace": {}, "identifiability": {"answer_unique": False}}
    query_value = int(query_match.group(1), 2)
    ident = identify_bit_expression(pairs, query_value, max_depth=2)
    if not ident["answer_unique"]:
        return {"status": "ambiguous", "answer": None, "trace": {"solver": "finite_semantic_enumeration", "identifiability": ident}, "identifiability": ident}
    answer = ident["query_outputs"][0]
    return {
        "status": "solved",
        "answer": answer,
        "trace": {"solver": "finite_semantic_enumeration", "expression": ident["selected_expression"], "example_inputs": [v for v, _ in pairs], "query_input": query_value, "identifiability": ident},
        "identifiability": ident,
    }


### 2.14 Bit 单次候选 Generator

In [ ]:
def generate_bit_sample(rng: np.random.Generator, examples: int = 5) -> dict[str, object]:
    """Generator: exactly one random DSL candidate; no identifiability retry loop."""
    tree = random_bit_tree(rng, depth=2)
    inputs = rng.choice(np.arange(256), size=examples + 1, replace=False).tolist()
    pairs = [(int(value), _bit_tree_value(tree, int(value))) for value in inputs[:-1]]
    query_value = int(inputs[-1])
    prompt = (
        "Below are input-output examples for a hidden deterministic 8-bit transformation.\n"
        + "\n".join(f"Input: 0b{value:08b} -> Output: 0b{output:08b}" for value, output in pairs)
        + f"\nNow, determine the output for input: 0b{query_value:08b}"
    )
    expression = render_bit_tree(tree)
    return {
        "task_type": "bit_manipulation",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"expression": expression, "depth": 2, "width": 8, "operations": sorted(set(re.findall(r"\b(?:SHL|SHR|ROTL|ROTR|AND|OR|XOR|NOT)\b", expression)))},
        "prompt": prompt,
        "answer": "",
        "trace": {},
        "difficulty": "depth_2",
        "deduce_or_guess": "deduce",
        "generation_status": "generated",
        "generation_error": None,
    }


### 2.15 Bit 独立 Verifier：解析、求值和语义去重

In [ ]:
def _verify_bit_value(node: tuple, value: int) -> int:
    kind = node[0]
    if kind == "var": return int(value) & MASK8
    if kind == "lit": return int(node[1]) & MASK8
    if kind in BIT_SHIFT_OPS:
        child, shift = _verify_bit_value(node[1], value), int(node[2]) % 8
        if kind == "SHL": return (child << shift) & MASK8
        if kind == "SHR": return (child >> shift) & MASK8
        if kind == "ROTL": return ((child << shift) | (child >> (8 - shift))) & MASK8 if shift else child
        return ((child >> shift) | (child << (8 - shift))) & MASK8 if shift else child
    if kind == "NOT": return (~_verify_bit_value(node[1], value)) & MASK8
    left, right = _verify_bit_value(node[1], value), _verify_bit_value(node[2], value)
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def _verify_parse_bit_expression(text: str, position: int = 0) -> tuple[tuple, int]:
    while position < len(text) and text[position].isspace(): position += 1
    if text.startswith("x", position): return ("var",), position + 1
    literal = re.match(r"0b([01]{8})", text[position:])
    if literal: return ("lit", int(literal.group(1), 2)), position + literal.end()
    operation = re.match(r"(NOT|SHL|SHR|ROTL|ROTR|AND|OR|XOR)\(", text[position:])
    if operation is None: raise ValueError("invalid_bit_expression")
    op, pos = operation.group(1), position + operation.end()
    left, pos = _verify_parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if op == "NOT":
        if pos >= len(text) or text[pos] != ")": raise ValueError("invalid_not")
        return (op, left), pos + 1
    if pos >= len(text) or text[pos] != ",": raise ValueError("missing_comma")
    pos += 1
    if op in BIT_SHIFT_OPS:
        match = re.match(r"\s*([0-7])\s*\)", text[pos:])
        if match is None: raise ValueError("invalid_shift")
        return (op, left, int(match.group(1))), pos + match.end()
    right, pos = _verify_parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if pos >= len(text) or text[pos] != ")": raise ValueError("missing_close")
    return (op, left, right), pos + 1


def _verify_evaluate_bit_expression(text: str, value: int) -> int:
    tree, position = _verify_parse_bit_expression(text)
    if text[position:].strip(): raise ValueError("trailing_expression")
    return _verify_bit_value(tree, value)


def _verify_bit_identifiability(example_pairs: list[tuple[int, int]], query_value: int, max_depth: int = 2) -> dict[str, object]:
    matching = []
    for tree in _bit_candidate_trees(max_depth):
        if all(_verify_bit_value(tree, value) == output for value, output in example_pairs):
            matching.append(tree)
    semantic_groups = {}
    for tree in matching:
        signature = tuple(_verify_bit_value(tree, value) for value in range(256))
        semantic_groups.setdefault(signature, tree)
    query_outputs = sorted({_verify_bit_value(tree, query_value) for tree in semantic_groups.values()})
    return {
        "raw_candidate_count": len(matching),
        "semantic_candidate_count": len(semantic_groups),
        "query_output_count": len(query_outputs),
        "query_outputs": [f"{value:08b}" for value in query_outputs],
        "expression_unique": len(matching) == 1,
        "answer_unique": len(query_outputs) == 1,
    }


### 2.16 Bit 样本 Verifier 与可识别性判定

In [ ]:
def verify_bit_sample(sample: dict[str, object]) -> dict[str, object]:
    prompt = str(sample.get("prompt", ""))
    pairs = [(int(value, 2), int(output, 2)) for value, output in re.findall(r"Input: 0b([01]{8}) -> Output: 0b([01]{8})", prompt)]
    query_match = re.search(r"output for input: 0b([01]{8})", prompt)
    trace = sample.get("trace", {})
    answer = str(sample.get("answer", "")).removeprefix("0b")
    ident = _verify_bit_identifiability(pairs, int(query_match.group(1), 2), max_depth=2) if query_match else {"query_output_count": 0, "answer_unique": False, "expression_unique": False, "query_outputs": []}
    answer_ok = bool(query_match and pairs and re.fullmatch(r"[01]{8}", answer) and ident.get("answer_unique") and answer in ident.get("query_outputs", []))
    trace_ok = isinstance(trace, dict) and trace.get("solver") == "finite_semantic_enumeration"
    if answer_ok:
        try:
            expression = str(trace["expression"])
            answer_ok = all(_verify_evaluate_bit_expression(expression, value) == output for value, output in pairs)
            answer_ok = answer_ok and _verify_evaluate_bit_expression(expression, int(query_match.group(1), 2)) == int(answer, 2)
            trace_ok = trace_ok and trace.get("example_inputs") == [v for v, _ in pairs] and trace.get("query_input") == int(query_match.group(1), 2)
        except (KeyError, ValueError, TypeError, IndexError):
            answer_ok = False
            trace_ok = False
    return {"answer_ok": bool(answer_ok), "trace_ok": bool(trace_ok), "identifiability": ident, "difficulty_ok": str(sample.get("difficulty", "")).startswith("depth_"), "solver_status": "independent_parser_and_finite_enumeration"}


### 2.17 Equation Numeric：操作数解析、算术和输出格式

In [ ]:
# Equation Numeric: 24-rule prior, independent solver, and deterministic verifier.
NUMERIC_EXPRESSION_RE = re.compile(r"^\d{2}[^\d]\d{2}$")
numeric_operator_pool = train_numeric_operator_pool or tuple("+-*/|\\@$%&!?^#~`")


def _numeric_operands(expression: str, parse_name: str) -> tuple[str, str]:
    if NUMERIC_EXPRESSION_RE.fullmatch(expression) is None:
        raise ValueError("invalid_numeric_expression")
    left_text, right_text = expression[:2], expression[3:]
    if parse_name == "AB_CD":
        return left_text, right_text
    if parse_name == "BA_DC":
        return left_text[::-1], right_text[::-1]
    raise ValueError("unsupported_parse")


def _numeric_operation(operation: str, left_text: str, right_text: str) -> int | str | None:
    left, right = int(left_text), int(right_text)
    if operation == "concat_lr": return left_text + right_text
    if operation == "concat_rl": return right_text + left_text
    if operation == "add": return left + right
    if operation == "add_p1": return left + right + 1
    if operation == "add_m1": return left + right - 1
    if operation == "sub_lr": return left - right
    if operation == "sub_rl": return right - left
    if operation == "abs_sub": return abs(left - right)
    if operation == "mul": return left * right
    if operation == "mul_p1": return left * right + 1
    if operation == "mul_m1": return left * right - 1
    if operation == "max_mod_min":
        smaller = min(left, right)
        return None if smaller == 0 else max(left, right) % smaller
    raise ValueError(f"unsupported_operation:{operation}")


def _numeric_sign_token(operator: str) -> str:
    return "-" if operator == "-" else operator


def _numeric_format(value: int | str, format_name: str, operator: str) -> str | None:
    if format_name == "raw":
        if isinstance(value, str): return value
        return str(value) if value >= 0 else f"{_numeric_sign_token(operator)}{abs(value)}"
    if not isinstance(value, int): return None
    if format_name == "rev": return str(value)[::-1] if value >= 0 else None
    if format_name == "sign_prefix_rev": return str(value)[::-1] if value >= 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "zpad2": return f"{_numeric_sign_token(operator)}{abs(value):02d}" if value < 0 else f"{value:02d}"
    if format_name == "neg_prefix_rev": return "0" if value == 0 else (None if value > 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}")
    if format_name == "always_prefix_rev": return "0" if value == 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "sign_prefix_raw": return "0" if value == 0 else f"{operator}{abs(value)}"
    if format_name == "sign_prefix_zpad2": return "00" if value == 0 else f"{operator}{abs(value):02d}"
    if format_name == "neg_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "always_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "rev_or_neg_suffix": return str(value)[::-1] if value >= 0 else f"{str(abs(value))[::-1]}{operator}"
    raise ValueError(f"unsupported_format:{format_name}")


def numeric_rule_output(rule_name: str, expression: str) -> str | None:
    spec = NUMERIC24_RULE_BY_NAME[rule_name]
    left_text, right_text = _numeric_operands(expression, spec["parse"])
    operation_value = _numeric_operation(spec["operation"], left_text, right_text)
    return None if operation_value is None else _numeric_format(operation_value, spec["format"], expression[2])


### 2.18 Equation Numeric Solver：候选规则与全局约束

In [ ]:
def _parse_numeric_prompt(prompt: str) -> tuple[list[tuple[str, str]], str] | None:
    examples, query = [], None
    for line in str(prompt).splitlines():
        line = line.strip()
        if line.startswith("Now, determine the result for:"):
            query = line.split(":", 1)[1].strip()
        elif " = " in line:
            expression, output = line.split(" = ", 1)
            if NUMERIC_EXPRESSION_RE.fullmatch(expression.strip()): examples.append((expression.strip(), output.strip()))
    return (examples, query) if examples and query else None


def _numeric_candidate_rules(examples: list[tuple[str, str]], operator: str) -> list[str]:
    same_operator = [(expression, output) for expression, output in examples if expression[2] == operator]
    candidates = []
    for rule in NUMERIC24_RULES:
        if all(numeric_or_text_match(numeric_rule_output(rule["name"], expression), output) for expression, output in same_operator):
            candidates.append(rule["name"])
    return candidates


def _numeric_valid_assignments(candidates_by_operator: dict[str, list[str]], operators: list[str]) -> list[dict[str, str]]:
    assignments = []
    for mode in ("normal", "flip"):
        selected = {}
        used_groups = set()
        def backtrack(index: int) -> None:
            if index == len(operators):
                assignments.append({"mode": mode, "rules": dict(selected)})
                return
            operator = operators[index]
            for rule_name in candidates_by_operator[operator]:
                spec = NUMERIC24_RULE_BY_NAME[rule_name]
                if spec["family"] != "join" and spec["family"] != mode:
                    continue
                group = spec["group"]
                if group != "join_like" and group in used_groups:
                    continue
                selected[operator] = rule_name
                if group != "join_like": used_groups.add(group)
                backtrack(index + 1)
                if group != "join_like": used_groups.remove(group)
                selected.pop(operator, None)
        backtrack(0)
    return assignments


def solve_equation_numeric_prompt(prompt: str) -> dict[str, object]:
    parsed = _parse_numeric_prompt(prompt)
    if parsed is None: return {"status": "invalid_prompt", "query_outputs": []}
    examples, query_expression = parsed
    query_operator = query_expression[2]
    example_operators = []
    for expression, _ in examples:
        if expression[2] not in example_operators: example_operators.append(expression[2])
    operators = example_operators + ([] if query_operator in example_operators else [query_operator])
    candidates_by_operator = {operator: _numeric_candidate_rules(examples, operator) for operator in operators}
    if any(not values for values in candidates_by_operator.values()):
        return {"status": "no_compatible_rule", "query_outputs": [], "candidates_by_operator": candidates_by_operator}
    assignments = _numeric_valid_assignments(candidates_by_operator, operators)
    query_outputs = []
    for assignment in assignments:
        value = numeric_rule_output(assignment["rules"][query_operator], query_expression)
        if value is not None and not any(numeric_or_text_match(value, previous) for previous in query_outputs):
            query_outputs.append(value)
    query_outputs.sort()
    selected = assignments[0] if assignments else None
    return {
        "status": "solved" if assignments else "no_valid_global_assignment",
        "query_operator": query_operator,
        "query_operator_in_examples": query_operator in example_operators,
        "candidates_by_operator": candidates_by_operator,
        "valid_assignment_count": len(assignments),
        "valid_assignments": assignments[:20],
        "query_outputs": query_outputs,
        "answer_unique": len(query_outputs) == 1,
        "selected_assignment": selected,
    }


### 2.19 Equation Numeric Generator：真值计算与题面生成

In [ ]:
def _numeric_truth_output(rule_name: str, expression: str) -> str | None:
    """Generator-side arithmetic implementation kept separate from the solver API."""
    spec = NUMERIC24_RULE_BY_NAME[rule_name]
    left, right = expression[:2], expression[3:]
    if spec["parse"] == "BA_DC": left, right = left[::-1], right[::-1]
    a, b = int(left), int(right)
    operation = spec["operation"]
    if operation == "concat_lr": value = left + right
    elif operation == "concat_rl": value = right + left
    elif operation == "add": value = a + b
    elif operation == "add_p1": value = a + b + 1
    elif operation == "add_m1": value = a + b - 1
    elif operation == "sub_lr": value = a - b
    elif operation == "sub_rl": value = b - a
    elif operation == "abs_sub": value = abs(a - b)
    elif operation == "mul": value = a * b
    elif operation == "mul_p1": value = a * b + 1
    elif operation == "mul_m1": value = a * b - 1
    elif operation == "max_mod_min":
        value = None if min(a, b) == 0 else max(a, b) % min(a, b)
    else: raise ValueError(operation)
    return None if value is None else _numeric_format(value, spec["format"], expression[2])


### 2.20 Equation Numeric Generator：采样辅助函数

In [ ]:
def _numeric_random_expression(rng: np.random.Generator, operator: str, avoid_zero_divisor: bool = False) -> str:
    left, right = int(rng.integers(0, 100)), int(rng.integers(0, 100))
    if avoid_zero_divisor and right == 0: right = int(rng.integers(1, 100))
    return f"{left:02d}{operator}{right:02d}"


def _numeric_prompt_text(example_pairs: list[tuple[str, str]], query_expression: str) -> str:
    return (
        "In this numeric equation task, each operator has a hidden transformation rule.\n"
        "The same rule assignment is used consistently within this prompt.\n"
        + "\n".join(f"{expression} = {output}" for expression, output in example_pairs)
        + f"\nNow, determine the result for: {query_expression}"
    )


def _numeric_hidden_assignment(rng: np.random.Generator, mode: str) -> dict[str, str]:
    pools = {}
    for group in ("sub_or_mod_like", "add_like", "mul_like"):
        choices = [rule["name"] for rule in NUMERIC24_RULES if rule["family"] == mode and rule["group"] == group]
        pools[group] = str(rng.choice(choices))
    pools["join_like"] = str(rng.choice(["join", "join_swap"]))
    return pools


### 2.21 Equation Numeric Generator：生成单次候选

In [ ]:
def generate_equation_numeric_sample(
    rng: np.random.Generator,
    guess: bool = False,
    guess_variant: str | None = None,
) -> dict[str, object]:
    """Generator: exactly one multi-operator candidate, with no retry loop."""
    mode = str(rng.choice(["normal", "flip"]))
    by_group = _numeric_hidden_assignment(rng, mode)
    if guess:
        example_operators = list(rng.choice(np.asarray(numeric_operator_pool), size=3, replace=False))
        query_operator = next(operator for operator in numeric_operator_pool if operator not in example_operators)
        hidden = {operator: by_group[group] for operator, group in zip(example_operators, ("sub_or_mod_like", "add_like", "mul_like"))}
        hidden[query_operator] = by_group["join_like"]
    else:
        operators = list(rng.choice(np.asarray(numeric_operator_pool), size=4, replace=False))
        hidden = {operator: by_group[group] for operator, group in zip(operators, ("sub_or_mod_like", "add_like", "mul_like", "join_like"))}
        query_operator = str(rng.choice(np.asarray(operators)))
        example_operators = operators
    pairs = []
    for operator in example_operators:
        rule_name = hidden[operator]
        for _example in range(5):
            expression = _numeric_random_expression(rng, operator, avoid_zero_divisor="mod" in NUMERIC24_RULE_BY_NAME[rule_name]["operation"])
            output = _numeric_truth_output(rule_name, expression)
            if output is not None:
                pairs.append((expression, output))
    if guess and guess_variant == "guess_answer_equivalent":
        same = int(rng.integers(10, 100))
        query_expression = f"{same:02d}{query_operator}{same:02d}"
    elif guess:
        left = int(rng.integers(10, 100))
        right = int(rng.integers(10, 100))
        if right == left:
            right = (right + 1) % 90 + 10
        query_expression = f"{left:02d}{query_operator}{right:02d}"
    else:
        query_expression = _numeric_random_expression(rng, query_operator, avoid_zero_divisor=False)
    return {
        "task_type": "equation_numeric",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"mode": mode, "operator_to_rule": hidden, "rule_formulas": {name: NUMERIC24_FORMULAS[name] for name in hidden.values()}},
        "prompt": _numeric_prompt_text(pairs, query_expression),
        "answer": "",
        "trace": {},
        "difficulty": "guess_candidate" if guess else "deduce_multi_operator",
        "deduce_or_guess": "guess" if guess else "deduce",
        "guess_subtype": "pending" if guess else "deduce",
        "generation_status": "generated",
        "generation_error": None,
    }


### 2.22 Equation Numeric Verifier：答案格式与候选规则

In [ ]:
def _numeric_verifier_format(value: int | str, format_name: str, operator: str) -> str | None:
    """Verifier-only formatter; kept separate from generator and Solver formatters."""
    if format_name == "raw":
        if isinstance(value, str): return value
        return str(value) if value >= 0 else f"{_numeric_sign_token(operator)}{abs(value)}"
    if not isinstance(value, int): return None
    if format_name == "rev": return str(value)[::-1] if value >= 0 else None
    if format_name == "sign_prefix_rev": return str(value)[::-1] if value >= 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "zpad2": return f"{_numeric_sign_token(operator)}{abs(value):02d}" if value < 0 else f"{value:02d}"
    if format_name == "neg_prefix_rev": return "0" if value == 0 else (None if value > 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}")
    if format_name == "always_prefix_rev": return "0" if value == 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "sign_prefix_raw": return "0" if value == 0 else f"{operator}{abs(value)}"
    if format_name == "sign_prefix_zpad2": return "00" if value == 0 else f"{operator}{abs(value):02d}"
    if format_name == "neg_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "always_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "rev_or_neg_suffix": return str(value)[::-1] if value >= 0 else f"{str(abs(value))[::-1]}{operator}"
    raise ValueError(f"unsupported_format:{format_name}")


def _numeric_verify_output(rule_name: str, expression: str) -> str | None:
    """Verifier-side arithmetic and formatting implementation."""
    spec = NUMERIC24_RULE_BY_NAME.get(rule_name)
    if spec is None or NUMERIC_EXPRESSION_RE.fullmatch(expression) is None:
        return None
    left_text, right_text = expression[:2], expression[3:]
    if spec["parse"] == "BA_DC": left_text, right_text = left_text[::-1], right_text[::-1]
    left, right = int(left_text), int(right_text)
    op = spec["operation"]
    if op == "concat_lr": value = left_text + right_text
    elif op == "concat_rl": value = right_text + left_text
    elif op == "add": value = left + right
    elif op == "add_p1": value = left + right + 1
    elif op == "add_m1": value = left + right - 1
    elif op == "sub_lr": value = left - right
    elif op == "sub_rl": value = right - left
    elif op == "abs_sub": value = abs(left - right)
    elif op == "mul": value = left * right
    elif op == "mul_p1": value = left * right + 1
    elif op == "mul_m1": value = left * right - 1
    elif op == "max_mod_min": value = None if min(left, right) == 0 else max(left, right) % min(left, right)
    else: return None
    return None if value is None else _numeric_verifier_format(value, spec["format"], expression[2])


def _numeric_verifier_candidate_rules(examples: list[tuple[str, str]], operator: str) -> list[str]:
    same_operator = [(expression, output) for expression, output in examples if expression[2] == operator]
    candidates = []
    for rule in NUMERIC24_RULES:
        if all(numeric_or_text_match(_numeric_verify_output(rule["name"], expression), output) for expression, output in same_operator):
            candidates.append(rule["name"])
    return candidates


def _numeric_verifier_valid_assignments(candidates_by_operator: dict[str, list[str]], operators: list[str]) -> list[dict[str, str]]:
    assignments = []
    for mode in ("normal", "flip"):
        selected, used_groups = {}, set()
        def backtrack(index: int) -> None:
            if index == len(operators):
                assignments.append({"mode": mode, "rules": dict(selected)})
                return
            operator = operators[index]
            for rule_name in candidates_by_operator.get(operator, []):
                spec = NUMERIC24_RULE_BY_NAME[rule_name]
                if spec["family"] != "join" and spec["family"] != mode:
                    continue
                group = spec["group"]
                if group != "join_like" and group in used_groups:
                    continue
                selected[operator] = rule_name
                if group != "join_like": used_groups.add(group)
                backtrack(index + 1)
                if group != "join_like": used_groups.remove(group)
                selected.pop(operator, None)
        backtrack(0)
    return assignments


### 2.23 Equation Numeric Verifier：Prompt 与样本审计

In [ ]:
def verify_equation_numeric_prompt(prompt: str) -> dict[str, object]:
    """Independent Verifier path; does not call solve_equation_numeric_prompt."""
    parsed = _parse_numeric_prompt(prompt)
    if parsed is None:
        return {"status": "invalid_prompt", "query_outputs": [], "valid_assignment_count": 0}
    examples, query_expression = parsed
    query_operator = query_expression[2]
    example_operators = []
    for expression, _ in examples:
        if expression[2] not in example_operators:
            example_operators.append(expression[2])
    operators = example_operators + ([] if query_operator in example_operators else [query_operator])
    candidates_by_operator = {operator: _numeric_verifier_candidate_rules(examples, operator) for operator in operators}
    if any(not values for values in candidates_by_operator.values()):
        return {"status": "no_compatible_rule", "query_outputs": [], "candidates_by_operator": candidates_by_operator, "valid_assignment_count": 0}
    assignments = _numeric_verifier_valid_assignments(candidates_by_operator, operators)
    query_outputs = []
    for assignment in assignments:
        value = _numeric_verify_output(assignment["rules"][query_operator], query_expression)
        if value is not None and not any(numeric_or_text_match(value, previous) for previous in query_outputs):
            query_outputs.append(value)
    query_outputs.sort()
    return {
        "status": "solved" if assignments else "no_valid_global_assignment",
        "query_operator": query_operator,
        "query_operator_in_examples": query_operator in example_operators,
        "candidates_by_operator": candidates_by_operator,
        "valid_assignment_count": len(assignments),
        "valid_assignments": assignments[:20],
        "query_outputs": query_outputs,
        "answer_unique": len(query_outputs) == 1,
        "selected_assignment": assignments[0] if assignments else None,
    }


def verify_equation_numeric_sample(sample: dict[str, object]) -> dict[str, object]:
    parsed = _parse_numeric_prompt(str(sample.get("prompt", "")))
    if parsed is None:
        return {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False, "guess_subtype": "invalid"}
    examples, query_expression = parsed
    answer = str(sample.get("answer", ""))
    resolution = verify_equation_numeric_prompt(str(sample.get("prompt", "")))
    ident = {
        "valid_assignment_count": resolution.get("valid_assignment_count", 0),
        "query_output_count": len(resolution.get("query_outputs", [])),
        "query_outputs": resolution.get("query_outputs", []),
        "answer_unique": bool(resolution.get("answer_unique", False)),
        "rule_assignment_unique": resolution.get("valid_assignment_count", 0) == 1,
    }
    if query_expression[2] not in {expression[2] for expression, _ in examples}:
        if ident["rule_assignment_unique"]:
            guess_subtype = "guess_rule_unique"
        elif ident["answer_unique"]:
            guess_subtype = "guess_answer_equivalent"
        else:
            guess_subtype = "guess_answer_ambiguous"
    else:
        guess_subtype = "deduce"
    trace = sample.get("trace", {})
    trace_ok = isinstance(trace, dict) and trace.get("query_outputs") == resolution.get("query_outputs")
    answer_ok = bool(resolution.get("answer_unique", False) and any(numeric_or_text_match(answer, output) for output in resolution.get("query_outputs", [])))
    if guess_subtype == "guess_answer_ambiguous":
        answer_ok = False
    return {
        "answer_ok": answer_ok,
        "trace_ok": bool(trace_ok),
        "identifiability": ident,
        "guess_subtype": guess_subtype,
        "difficulty_ok": str(sample.get("difficulty", "")).startswith(("deduce_", "guess_")),
        "solver_status": "independent_24_rule_verifier",
    }


### 2.24 Screening Schema 与拒绝原因

Screening 统一检查 Schema、答案、轨迹、可识别性、难度和重复；所有拒绝原因都会保留。

In [ ]:
SYNTHETIC_VERIFIER_VERSION = "synthetic_independent_verifier_v4"

# Unified screening, export, leakage audit, and optional local-vLLM Data Designer Preview.
import copy

SYNTHETIC_SCHEMA_FIELDS = [
    "candidate_id", "task_type", "source", "generator_version", "hidden_rule", "prompt", "answer", "trace", "difficulty",
    "deduce_or_guess", "generation_status", "generation_error",
]


def synthetic_prompt_fingerprint(text: object) -> str:
    return prompt_hash(normalize_prompt(text))


def _screen_verification(sample: dict[str, object]) -> dict[str, object]:
    if sample.get("task_type") == "numeral_system": return verify_numeral_sample(sample)
    if sample.get("task_type") == "gravity": return verify_gravity_sample(sample)
    if sample.get("task_type") == "bit_manipulation": return verify_bit_sample(sample)
    if sample.get("task_type") == "equation_numeric": return verify_equation_numeric_sample(sample)
    return {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False}


def screen_synthetic_sample(sample: dict[str, object], seen_exact: set[str], seen_normalized: set[str], seen_structure: set[str], leakage: dict[str, object]) -> dict[str, object]:
    reasons = []
    required_fields = [field for field in SYNTHETIC_SCHEMA_FIELDS if field != "generation_error"]
    schema_ok = all(field in sample for field in required_fields) and "generation_error" in sample
    schema_ok = schema_ok and isinstance(sample.get("prompt"), str) and isinstance(sample.get("trace"), dict)
    if not schema_ok: reasons.append("schema_invalid")
    if sample.get("generation_status") != "generated": reasons.append("generation_failed")
    if sample.get("task_type") not in TARGET_TASKS: reasons.append("unsupported_task")
    if sample.get("source") != "deterministic_prior_generator": reasons.append("unexpected_source")
    exact_key = prompt_hash(sample.get("prompt", ""))
    normalized_key = synthetic_prompt_fingerprint(sample.get("prompt", ""))
    structure_key = structure_fingerprint(sample.get("prompt", ""), str(sample.get("task_type", "")))
    if exact_key in seen_exact: reasons.append("duplicate_prompt")
    if normalized_key in seen_normalized: reasons.append("near_duplicate_prompt")
    structure_duplicate = structure_key in seen_structure
    try:
        verification = _screen_verification(sample)
    except (ValueError, TypeError, KeyError, IndexError, ZeroDivisionError) as exc:
        verification = {}
        reasons.append(f"verifier_error:{type(exc).__name__}")
    if not verification.get("answer_ok", False): reasons.append("answer_verification_failed")
    if not verification.get("trace_ok", False): reasons.append("trace_verification_failed")
    ident = verification.get("identifiability", {})
    if not ident.get("answer_unique", False): reasons.append("answer_not_identifiable")
    if not verification.get("difficulty_ok", False): reasons.append("difficulty_audit_failed")
    guess_subtype = verification.get("guess_subtype", sample.get("guess_subtype", "deduce"))
    if sample.get("task_type") == "equation_numeric":
        sample["guess_subtype"] = guess_subtype
        if guess_subtype == "guess_answer_ambiguous": reasons.append("guess_answer_ambiguous")
    reasons.extend(leakage["rejection_reasons"])
    accepted = not reasons
    record = {
        "candidate_id": sample.get("candidate_id", sample.get("candidate_index", "")),
        "candidate_index": sample.get("candidate_index", -1),
        "task_type": sample.get("task_type", ""),
        "source": sample.get("source", ""),
        "generator_version": sample.get("generator_version", ""),
        "generation_status": sample.get("generation_status", ""),
        "generation_error": sample.get("generation_error"),
        "deduce_or_guess": sample.get("deduce_or_guess", "unknown"),
        "guess_subtype": guess_subtype,
        "prompt_hash": exact_key,
        "normalized_prompt_hash": normalized_key,
        "structure_fingerprint": structure_key,
        "structure_duplicate_within_synthetic": structure_duplicate,
        "schema_ok": schema_ok,
        "answer_ok": bool(verification.get("answer_ok", False)),
        "trace_ok": bool(verification.get("trace_ok", False)),
        "identifiability": json.dumps(ident, ensure_ascii=False, sort_keys=True),
        "difficulty": sample.get("difficulty", ""),
        "leakage_status": leakage["status"],
        "numeral_query_exposure_status": leakage["numeral_query_exposure"]["status"],
        "leakage_matches": json.dumps(leakage["matches"], ensure_ascii=False),
        "accepted": accepted,
        "verification_status": "accepted" if accepted else "rejected",
        "verifier_version": SYNTHETIC_VERIFIER_VERSION,
        "rejection_reasons": ";".join(dict.fromkeys(reasons)),
    }
    return record


def _write_jsonl(path: Path, records: list[dict[str, object]]) -> None:
    payload = "\n".join(json.dumps(record, ensure_ascii=False) for record in records)
    path.write_text(payload + ("\n" if payload else ""), encoding="utf-8")


## 3. 与三个真实集合交叉审计

精确 Prompt、规范化 Prompt 和来源 ID 命中表示真实重复；命中任一真实集合都会拒绝，
其中 Validation / Holdout 的重复或增强来源绝不进入 Accepted。
结构相似单独记录，不作为拒绝依据。候选未声明来源 ID 时记录为 unavailable，不伪造来源。
没有通过审计的历史 Accepted 不会被自动沿用。

Numeral 还需单独完成下方 Query 映射暴露审计；`leakage_status=passed` 只表示通过样本重复/来源检查，不能解释为映射未暴露。

### 3.1 Numeral Query 映射暴露

以 `(阿拉伯数字, 规范罗马数字)` 为键，分别检查：合成 Query 是否出现在真实示例中、是否与真实目标相同，以及真实 Query 是否出现在合成示例中。
第三项反映合成训练内容对真实评测答案的反向暴露。题内 Query 已作为示例给出也单独记录。

这项审计与 00 一样只报告有限映射暴露，不改变 IID 划分或生成分布，也不据此剔除正确的标准转换。
Train / Validation / Holdout Test 的目标答案仅用于映射暴露审计，不用于生成、Solver、先验推断或调参；生成器只读取 Train Prompt 中的数字和示例数量。Numeral 解析或映射无效会使审计失败并拒绝该候选。


In [ ]:
def build_numeral_mapping_index(frame):
    index = {"examples": {}, "targets": {}}
    for row in frame[frame.task_type == "numeral_system"].itertuples(index=False):
        pairs, query = parse_numeral_prompt(row.prompt)
        if not verify_roman_number(query, row.answer) or not all(verify_roman_number(*pair) for pair in pairs):
            raise ValueError(f"真实 Numeral 映射与已清洗数据契约不符: {row.id}")
        reference = {"id": str(row.id), "split": row.split}
        for pair in sorted(set(pairs)):
            index["examples"].setdefault(pair, []).append(reference)
        index["targets"].setdefault((query, row.answer), []).append(reference)
    return index


def audit_numeral_query_exposure(sample, index):
    if sample.get("task_type") != "numeral_system":
        return {"status": "not_applicable"}
    try:
        pairs, query = parse_numeral_prompt(str(sample.get("prompt", "")))
        answer = str(sample.get("answer", ""))
        if not verify_roman_number(query, answer) or not all(verify_roman_number(*pair) for pair in pairs):
            raise ValueError("Query/示例不是有效的标准 Roman 映射")
    except (ValueError, TypeError, KeyError) as error:
        return {"status": "invalid", "error": str(error)}
    by_split = {}
    for name in ("train", "validation", "holdout_test"):
        by_split[name] = {
            "query_in_real_examples": [row for row in index["examples"].get((query, answer), []) if row["split"] == name],
            "same_query_in_real_targets": [row for row in index["targets"].get((query, answer), []) if row["split"] == name],
            "real_queries_in_synthetic_examples": [
                {**row, "query_number": number, "query_roman": roman}
                for number, roman in sorted(set(pairs))
                for row in index["targets"].get((number, roman), []) if row["split"] == name
            ],
        }
    return {"status": "audited", "query_number": query, "query_roman": answer,
            "query_already_in_own_examples": (query, answer) in pairs, "by_split": by_split}


In [ ]:
def source_ids(sample: dict[str, object]) -> set[str]:
    ids = set()
    for field in ("id", "source_id", "source_ids", "source_row_id", "original_id", "parent_id", "parent_ids", "origin_id"):
        value = sample.get(field)
        values = value if isinstance(value, (list, tuple, set)) else [value]
        ids.update(str(item) for item in values if item is not None and str(item))
    return ids


def build_real_index(frame: pd.DataFrame) -> dict:
    index = {kind: {} for kind in ("exact", "normalized", "source_id", "source_row", "structure")}
    for row in frame.itertuples(index=False):
        keys = {"exact": prompt_hash(row.prompt), "normalized": synthetic_prompt_fingerprint(row.prompt),
                "source_id": str(row.id), "source_row": str(row.source_row),
                "structure": structure_fingerprint(row.prompt, row.task_type)}
        reference = {"id": str(row.id), "split": row.split, "source_row": str(row.source_row)}
        for kind, key in keys.items():
            index[kind].setdefault(key, []).append(reference)
    index["numeral_mapping"] = build_numeral_mapping_index(frame)
    return index


def audit_synthetic_leakage(sample: dict[str, object], real_index: dict) -> dict[str, object]:
    prompt = sample.get("prompt", "")
    ids = source_ids(sample)
    keys = {"exact": {prompt_hash(prompt)}, "normalized": {synthetic_prompt_fingerprint(prompt)},
            "source_id": ids, "source_row": {str(sample["source_row"])} if sample.get("source_row") not in (None, "") else set(),
            "structure": {structure_fingerprint(prompt, str(sample.get("task_type", "")))}}
    matches = {kind: [row for key in values for row in real_index[kind].get(key, [])]
               for kind, values in keys.items()}
    reasons = sorted({f"real_{row['split']}_{kind}_duplicate" for kind in ("exact", "normalized", "source_id", "source_row")
                      for row in matches[kind]})
    exposure = audit_numeral_query_exposure(sample, real_index["numeral_mapping"])
    if exposure["status"] == "invalid": reasons.append("numeral_exposure_audit_failed")
    return {"status": "rejected" if reasons else "passed", "numeral_query_exposure": exposure, "matches": matches, "rejection_reasons": reasons,
            "lineage_status": "declared" if ids or keys["source_row"] else "unavailable",
            "structure_similar_only": bool(matches["structure"]) and not reasons}


real_index = build_real_index(split_df)


def _real_synthetic_cross_audit(candidates: list[dict[str, object]], destination=None) -> dict[str, object]:
    destination = Path(destination) if destination is not None else synthetic_dir
    destination.mkdir(parents=True, exist_ok=True)
    rows = []
    for sample in candidates:
        result = audit_synthetic_leakage(sample, real_index)
        rows.append({"candidate_id": sample.get("candidate_id", ""), "task_type": sample.get("task_type", ""), **result})
    pd.DataFrame(rows).to_csv(destination / "synthetic_real_leakage_audit.csv", index=False)
    summary = {
        "schema_version": "nemotron-synthetic-leakage/v3", "candidate_count": len(rows),
        "passed": sum(row["status"] == "passed" for row in rows),
        "rejected": sum(row["status"] == "rejected" for row in rows),
        "source_split_counts": split_df.groupby("split").size().to_dict(),
        "by_split": {name: {kind: sum(any(match["split"] == name for match in row["matches"][kind]) for row in rows)
                            for kind in ("exact", "normalized", "source_id", "source_row", "structure")}
                     for name in ("train", "validation", "holdout_test")},
        "dataset_manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        "split_sha256": {name: dataset_manifest["artifacts"][name]["sha256"] for name in dataset_manifest["files"]},
        "policy": "real duplicate or declared real lineage rejected; structure similarity reported only",
        "real_target_answers_used_for_exposure_audit": True,
        "heldout_answer_usage": "Numeral mapping exposure audit only; never generation, solver inference, or tuning",
        "numeral_query_exposure": _write_numeral_exposure_audit(candidates, destination),
        "lineage_unavailable": sum(row["lineage_status"] == "unavailable" for row in rows),
    }
    (destination / "synthetic_real_leakage_audit.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary

### 3.2 映射暴露明细与汇总

逐候选保存三个真实集合的命中 ID、映射及题内暴露；分别汇总全部 Candidates 和最终 Accepted。
合成 Query 暴露率的分母为有效合成 Query 数；真实 Query 覆盖率的分母为对应集合的真实 Numeral 样本数。
没有有效 Query 时比率记为 `null`，不写成零暴露。Accepted Manifest 绑定明细与汇总的 SHA256。


In [ ]:
def _write_numeral_exposure_audit(candidates, destination=None):
    destination = Path(destination) if destination is not None else synthetic_dir
    numeral_samples = [sample for sample in candidates if sample.get("task_type") == "numeral_system"]
    rows = []
    for sample in numeral_samples:
        exposure = sample["leakage_audit"]["numeral_query_exposure"]
        for name in ("train", "validation", "holdout_test"):
            matches = exposure.get("by_split", {}).get(name, {})
            rows.append({
                "candidate_id": sample.get("candidate_id", ""), "real_split": name,
                "accepted": sample.get("verification_status") == "accepted", "audit_status": exposure["status"],
                "query_number": exposure.get("query_number"), "query_roman": exposure.get("query_roman"),
                "query_already_in_own_examples": exposure.get("query_already_in_own_examples"),
                **{kind: json.dumps(matches.get(kind, []), ensure_ascii=False) for kind in
                   ("query_in_real_examples", "same_query_in_real_targets", "real_queries_in_synthetic_examples")},
                "audit_error": exposure.get("error", ""),
            })
    columns = ["candidate_id", "real_split", "accepted", "audit_status", "query_number", "query_roman",
               "query_already_in_own_examples", "query_in_real_examples", "same_query_in_real_targets",
               "real_queries_in_synthetic_examples", "audit_error"]
    pd.DataFrame(rows, columns=columns).to_csv(destination / "numeral_query_exposure.csv", index=False)
    populations = {"candidates": numeral_samples,
                   "accepted": [sample for sample in numeral_samples if sample.get("verification_status") == "accepted"]}
    summaries = {}
    real_counts = split_df[split_df.task_type == "numeral_system"].groupby("split").size().to_dict()
    for population, samples in populations.items():
        audits = [sample["leakage_audit"]["numeral_query_exposure"] for sample in samples]
        valid = [audit for audit in audits if audit["status"] == "audited"]
        by_split = {}
        for name in ("train", "validation", "holdout_test"):
            covered_ids = {row["id"] for audit in valid for row in audit["by_split"][name]["real_queries_in_synthetic_examples"]}
            exposed = sum(bool(audit["by_split"][name]["query_in_real_examples"]) for audit in valid)
            by_split[name] = {
                "synthetic_queries_audited": len(valid), "query_in_real_examples": exposed,
                "query_in_real_examples_rate": exposed / len(valid) if valid else None,
                "same_query_in_real_targets": sum(bool(audit["by_split"][name]["same_query_in_real_targets"]) for audit in valid),
                "real_queries_covered_by_synthetic_examples": len(covered_ids),
                "real_query_count": int(real_counts.get(name, 0)),
                "real_query_coverage_rate": len(covered_ids) / real_counts[name] if real_counts.get(name) else None,
            }
        summaries[population] = {"numeral_rows": len(samples), "audited": len(valid), "invalid": len(audits) - len(valid),
                                "query_already_in_own_examples": sum(audit["query_already_in_own_examples"] for audit in valid),
                                "by_split": by_split}
    summary = {
        "schema_version": "nemotron-synthetic-numeral-exposure/v1", "policy": "report_only",
        "interpretation": "finite mapping exposure; separate from full-sample duplicates and structural similarity",
        "claims_no_mapping_leakage": False, "populations": summaries,
        "dataset_manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        "split_sha256": {name: dataset_manifest["artifacts"][name]["sha256"] for name in dataset_manifest["files"]},
        "details_csv": "numeral_query_exposure.csv",
    }
    (destination / "numeral_query_exposure.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return summary


### 3.3 可选 NeMo Data Designer Preview

Data Designer 只做语言表达和轨迹变体，Seed 不包含答案、隐藏规则或教师轨迹。

In [ ]:
def _data_designer_preview() -> dict[str, object]:
    """Compatibility entrypoint; raw model traces are never automatically accepted."""
    if not run_data_designer_preview:
        return {"status": "skipped_by_default", "reason": "RUN_DATA_DESIGNER_PREVIEW=False"}
    if not (synthetic_dir / "data_designer/prepared_manifest.json").exists():
        return {"status": "blocked_unprepared_smoke", "reason": "先运行 utils/run_synthetic_smoke.py prepare，使用新的 Run ID"}
    return run_data_designer_smoke()


### 3.4 Solver 调度与 Pilot 生成计划

In [ ]:
def _solve_candidate(sample: dict[str, object]) -> dict[str, object]:
    task_type = sample.get("task_type")
    if task_type == "numeral_system":
        result = solve_numeral_prompt(str(sample.get("prompt", "")))
        sample["answer"] = result["answer"]
        sample["trace"] = result["trace"]
        sample["solver_status"] = result["status"]
        return result
    if task_type == "gravity":
        result = solve_gravity_prompt(str(sample.get("prompt", "")))
        sample["answer"] = result.get("answer") or ""
        sample["trace"] = result.get("trace", {})
        sample["solver_status"] = result.get("status", "unknown")
        return result
    if task_type == "equation_numeric":
        result = solve_equation_numeric_prompt(str(sample.get("prompt", "")))
        outputs = result.get("query_outputs", [])
        sample["answer"] = outputs[0] if result.get("answer_unique") and outputs else ""
        sample["trace"] = {
            "solver": "24-rule-global-assignment",
            "query_operator": result.get("query_operator"),
            "query_outputs": outputs,
            "valid_assignment_count": result.get("valid_assignment_count", 0),
            "candidate_rules_by_operator": result.get("candidates_by_operator", {}),
            "selected_assignment": result.get("selected_assignment"),
        }
        sample["solver_status"] = result.get("status", "unknown")
        return result
    if task_type == "bit_manipulation":
        result = solve_bit_prompt(str(sample.get("prompt", "")))
        sample["answer"] = result.get("answer") or ""
        sample["trace"] = result.get("trace", {})
        sample["solver_status"] = result.get("status", "unknown")
        return result
    return {"status": "unsupported_task", "answer": None}


def _generation_plan(task_type: str, index: int) -> dict[str, object]:
    if task_type == "equation_numeric" and index >= 80:
        variant = "guess_answer_equivalent" if index < 90 else "guess_answer_ambiguous"
        return {"guess": True, "guess_variant": variant}
    return {"guess": False, "guess_variant": None}


### 3.5 执行 Pilot、筛查并导出产物

In [ ]:
def run_synthetic_pilot(candidates: list[dict[str, object]] | None = None) -> dict[str, object]:
    rng = np.random.default_rng(SEED)
    generated, failures = [], []
    task_order = list(TASK_ORDER)
    if (synthetic_dir / "synthetic_candidates.jsonl").exists():
        raise FileExistsError("不覆盖既有候选；请设置新的 SYNTHETIC_RUN_ID")
    if candidates is None:
        for task_type in task_order:
            for index in range(pilot_candidates_per_task):
                candidate_id = f"{task_type}-{index:04d}"
                plan = _generation_plan(task_type, index)
                try:
                    if task_type == "numeral_system":
                        sample = generate_numeral_sample(rng)
                    elif task_type == "gravity":
                        sample = generate_gravity_sample(rng)
                    elif task_type == "equation_numeric":
                        sample = generate_equation_numeric_sample(rng, **plan)
                    else:
                        raise ValueError("非默认任务，不自动生成")
                    sample["candidate_id"] = candidate_id
                    sample["candidate_index"] = candidate_id
                    sample["requested_guess_variant"] = plan.get("guess_variant")
                    _solve_candidate(sample)
                    if task_type == "equation_numeric":
                        precheck = verify_equation_numeric_sample(sample)
                        sample["guess_subtype"] = precheck.get("guess_subtype", "pending")
                    generated.append(sample)
                except Exception as exc:
                    failure = {"candidate_id": candidate_id, "task_type": task_type, "generation_error": repr(exc)}
                    failures.append(failure)
                    generated.append({
                        "candidate_id": candidate_id, "candidate_index": candidate_id, "task_type": task_type,
                        "source": "deterministic_prior_generator", "generator_version": GENERATOR_VERSION,
                        "hidden_rule": {}, "prompt": "", "answer": "", "trace": {}, "difficulty": "generation_failed",
                        "deduce_or_guess": "unknown", "guess_subtype": "unknown", "generation_status": "generation_error",
                        "generation_error": repr(exc), "requested_guess_variant": plan.get("guess_variant"),
                    })
    else:
        generated = copy.deepcopy(candidates)
    seen_exact, seen_normalized, seen_structure = set(), set(), set()
    screening, accepted, rejected = [], [], []
    for sample in generated:
        leakage = audit_synthetic_leakage(sample, real_index)
        record = screen_synthetic_sample(sample, seen_exact, seen_normalized, seen_structure, leakage)
        screening.append(record)
        seen_exact.add(record["prompt_hash"])
        seen_normalized.add(record["normalized_prompt_hash"])
        seen_structure.add(record["structure_fingerprint"])
        sample["verification_status"] = record["verification_status"]
        sample["verifier_version"] = SYNTHETIC_VERIFIER_VERSION
        sample["leakage_status"] = leakage["status"]
        sample["leakage_audit"] = leakage
        sample["rejection_reasons"] = record["rejection_reasons"]
        (accepted if record["accepted"] else rejected).append(sample)
    _write_jsonl(synthetic_dir / "synthetic_candidates.jsonl", generated)
    _write_jsonl(synthetic_dir / "synthetic_accepted.jsonl", accepted)
    _write_jsonl(synthetic_dir / "synthetic_rejected.jsonl", rejected)
    screening_df = pd.DataFrame(screening)
    screening_df.to_csv(synthetic_dir / "synthetic_screening.csv", index=False)
    # 当前 Run 保存新审计结果，历史 Pilot 保持原样。
    cross_audit = _real_synthetic_cross_audit(generated)
    by_task = {}
    for task_type, group in screening_df.groupby("task_type", sort=True):
        reasons = Counter(reason for text in group["rejection_reasons"] for reason in str(text).split(";") if reason)
        by_task[task_type] = {"candidates": int(len(group)), "accepted": int(group["accepted"].sum()), "rejected": int((~group["accepted"]).sum()), "acceptance_rate": float(group["accepted"].mean()), "rejection_reasons": dict(reasons)}
    hidden_rule_counts, bit_operation_counts, gravity_values = Counter(), Counter(), []
    for sample in generated:
        hidden = sample.get("hidden_rule", {})
        if sample.get("task_type") == "equation_numeric": hidden_rule_counts.update(hidden.get("operator_to_rule", {}).values())
        elif sample.get("task_type") == "bit_manipulation": bit_operation_counts.update(hidden.get("operations", []))
        elif sample.get("task_type") == "gravity" and hidden.get("g") is not None: gravity_values.append(hidden["g"])
    data_designer_status = {"status": "not_run", "reason": "Run the separately prepared NeMo Smoke entrypoint"}
    metadata = {
        "schema_version": "nemotron-course-synthetic/v3", "seed": SEED, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_version": dataset_version, "dataset_manifest_sha256": cross_audit["dataset_manifest_sha256"],
        "requested_candidates_per_task": pilot_candidates_per_task, "task_order": task_order, "generator_version": GENERATOR_VERSION,
        "generator_is_one_shot": True, "train_prior_only": True, "validation_or_holdout_used_for_generation": False, "all_splits_used_for_leakage_audit": True,
        "generation_failures": failures, "data_designer": data_designer_status,
        "data_designer_official_api": {"package": "data-designer", "model_provider": "ModelProvider(provider_type='openai', endpoint='http://127.0.0.1:8000/v1', api_key=None)", "seed_api": "LocalFileSeedSource + DataDesignerConfigBuilder.with_seed_dataset", "preview_api": "DataDesigner.preview(config_builder=..., num_records=...)"},
        "raw_candidate_artifact": _synthetic_artifact_reference(synthetic_dir / "synthetic_candidates.jsonl"),
    }
    (synthetic_dir / "generation_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    guess_subtype_counts = {k: int(v) for k, v in screening_df["guess_subtype"].value_counts(dropna=False).to_dict().items()}
    statistics = {
        "mode": "pilot", "seed": SEED, "requested_candidates_per_task": pilot_candidates_per_task,
        "generated": len(generated), "accepted": len(accepted), "rejected": len(rejected), "generation_failures": len(failures),
        "by_task": by_task, "deduce_guess_counts": {k: int(v) for k, v in screening_df["deduce_or_guess"].value_counts(dropna=False).to_dict().items()},
        "deduce_guess_by_task": {task: {k: int(v) for k, v in group["deduce_or_guess"].value_counts(dropna=False).to_dict().items()} for task, group in screening_df.groupby("task_type", sort=True)},
        "guess_subtype_counts": guess_subtype_counts,
        "rule_coverage": {"equation_numeric": dict(hidden_rule_counts), "bit_operations": dict(bit_operation_counts), "gravity_g_unique": sorted(set(gravity_values))},
        "rejection_reasons": dict(Counter(reason for text in screening_df["rejection_reasons"] for reason in str(text).split(";") if reason)),
        "duplicate_audit": {"exact_duplicate_rows": int(len(screening_df) - screening_df["prompt_hash"].nunique()), "normalized_duplicate_rows": int(len(screening_df) - screening_df["normalized_prompt_hash"].nunique()), "structure_duplicate_rows": int(len(screening_df) - screening_df["structure_fingerprint"].nunique())},
        "real_cross_collection_audit": cross_audit,
        "data_designer": data_designer_status,
        "artifacts": {name: _synthetic_artifact_reference(synthetic_dir / name) for name in ["synthetic_candidates.jsonl", "synthetic_accepted.jsonl", "synthetic_rejected.jsonl", "synthetic_screening.csv", "synthetic_real_leakage_audit.json", "synthetic_real_leakage_audit.csv", "numeral_query_exposure.json", "numeral_query_exposure.csv", "generation_metadata.json"]},
    }
    (synthetic_dir / "synthetic_statistics.json").write_text(json.dumps(statistics, ensure_ascii=False, indent=2), encoding="utf-8")
    write_synthetic_accepted_manifest(accepted, cross_audit)
    return statistics

### 3.5 第一轮 NeMo / 本地 vLLM Smoke 的验证契约

每题型 3 条已独立验证且通过三集合 Audit 的 Generator Seed，共 9 条；每条请求两个 LLM 列，预计 18 个输出。公开 Seed 只包含 ID、题型和题面，不包含答案、隐藏规则或教师轨迹。

第一轮 Prompt Variant 只允许两个等价引导句之一加上**逐字保留的完整原题**，以便严格保留数字、运算符、单位、示例与 Query。暂不接受任意自由改写后仅靠正则能解析就声称语义不变。Trace Variant 是可以逐步复算的 JSON 证明，不包含无法验证的自由叙述；不完整、错误或无法解析的输出保留为 Preview。验证通过后写入独立的 `data_designer/accepted.jsonl`，不自动合并到后续 SFT。


In [ ]:
SMOKE_PREFIXES = ("Solve the task described below.", "Use the given examples to answer the query below.")
SMOKE_TASKS = ("numeral_system", "gravity", "equation_numeric")
SMOKE_VERIFIER_VERSION = "nemo-smoke-proof-v1"


def synthetic_accepted_status(accepted):
    if not accepted:
        return "empty_accepted"
    valid = all(row.get("verification_status") == "accepted" and row.get("leakage_status") == "passed" for row in accepted)
    return "passed" if valid else "failed_verification"


def verify_smoke_prompt(seed, text):
    checks = {"text": isinstance(text, str), "protected_math_unchanged": False}
    if not checks["text"]:
        return {"verified": False, "checks": checks}
    checks["protected_math_unchanged"] = text.strip() in {p + "\n" + seed["prompt"] for p in SMOKE_PREFIXES}
    if not checks["protected_math_unchanged"]:
        return {"verified": False, "checks": checks}
    candidate = copy.deepcopy(seed)
    candidate["prompt"] = text.strip()
    _solve_candidate(candidate)  # Public prompt only; do not retain a teacher trace as model output.
    result = _screen_verification(candidate)
    checks.update(answer_ok=bool(result.get("answer_ok")), trace_ok=bool(result.get("trace_ok")),
                  answer_unique=bool(result.get("identifiability", {}).get("answer_unique")),
                  original_answer_preserved=numeric_or_text_match(candidate["answer"], seed["answer"]))
    return {"verified": all(checks.values()), "checks": checks, "independent_verification": result,
            "solver_answer": candidate["answer"], "canonical_trace": candidate["trace"]}


In [ ]:
def _smoke_exact_output(actual, expected):
    # Trace fields must contain the rule's exact string result, including signs/zeroes.
    return type(actual) is str and type(expected) is str and actual == expected


def _smoke_equation_resolution(prompt):
    """Strict proof identifiability, independent of the legacy score tolerance."""
    examples, query = _parse_numeric_prompt(prompt)
    operators = list(dict.fromkeys([e[0][2] for e in examples] + [query[2]]))
    candidates = {op: [r["name"] for r in NUMERIC24_RULES
                      if all(_smoke_exact_output(_numeric_verify_output(r["name"], e), y)
                             for e, y in examples if e[2] == op)] for op in operators}
    assignments = _numeric_verifier_valid_assignments(candidates, operators)
    outputs = {_numeric_verify_output(a["rules"][query[2]], query) for a in assignments}
    outputs.discard(None)
    return {"answer_unique": len(outputs) == 1, "query_outputs": sorted(outputs),
            "selected_assignment": assignments[0] if assignments else None}


In [ ]:
def _smoke_close(value, expected):
    # More stringent than final-answer official 1% tolerance: verify intermediate computations.
    return isinstance(value, (float, int)) and not isinstance(value, bool) and math.isfinite(value) and math.isclose(value, float(expected), rel_tol=1e-4, abs_tol=1e-5)


def _verify_roman_proof(seed, proof):
    if set(proof) != {"task_type", "query", "components", "answer"}:
        return False
    examples, query = parse_numeral_prompt(seed["prompt"])
    sample = {**seed, "answer": proof["answer"], "trace": {"query_number": proof["query"], "components": proof["components"]}}
    result = verify_numeral_sample(sample)
    parts = proof["components"]
    return (proof["query"] == query and bool(parts) and
            all(set(p) == {"arabic", "roman"} and type(p["arabic"]) is int for p in parts) and
            result["answer_ok"] and result["trace_ok"] and proof["answer"] == seed["answer"])


def _verify_gravity_proof(seed, proof):
    if set(proof) != {"task_type", "g_estimates", "g_ols", "query_t", "query_distance", "answer"}:
        return False
    examples, query = _gravity_prompt_parts(seed["prompt"])
    fit = _gravity_verifier_fit(examples)
    estimates = proof["g_estimates"]
    return (fit["valid"] and len(estimates) == len(examples) and
            all(_smoke_close(a, b) for a, b in zip(estimates, fit["g_estimates"])) and
            _smoke_close(proof["g_ols"], fit["g_ols"]) and str(proof["query_t"]) == query and
            _smoke_close(proof["query_distance"], _gravity_verifier_distance(float(query), fit["g_ols"])) and
            official_numeric_match(proof["answer"], seed["answer"]))


def _verify_equation_proof(seed, proof):
    if set(proof) != {"task_type", "rules", "example_outputs", "query_output", "answer"}:
        return False
    examples, query = _parse_numeric_prompt(seed["prompt"])
    operators = list(dict.fromkeys([e[0][2] for e in examples] + [query[2]]))
    rules = proof["rules"]
    if set(rules) != set(operators) or any(r not in NUMERIC24_RULE_BY_NAME for r in rules.values()):
        return False
    assignments = _numeric_verifier_valid_assignments({op: [rules[op]] for op in operators}, operators)
    computed = [_numeric_verify_output(rules[e[2]], e) for e, _ in examples]
    resolution = _smoke_equation_resolution(seed["prompt"])
    return (bool(assignments) and isinstance(proof["example_outputs"], list) and len(proof["example_outputs"]) == len(examples) and
            all(_smoke_exact_output(a, b) and _smoke_exact_output(a, expected)
                for a, b, (_, expected) in zip(computed, proof["example_outputs"], examples)) and
            _smoke_exact_output(_numeric_verify_output(rules[query[2]], query), proof["query_output"]) and
            _smoke_exact_output(proof["query_output"], proof["answer"]) and
            numeric_or_text_match(proof["answer"], seed["answer"]) and resolution["answer_unique"])


def verify_smoke_trace(seed, text):
    try:
        # No JSON repair or extraction from prose: unverified surrounding claims stay Preview.
        def unique_object(pairs):
            result = {}
            for key, value in pairs:
                if key in result: raise ValueError("duplicate JSON key")
                result[key] = value
            return result
        proof = json.loads(text, object_pairs_hook=unique_object, parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
        if not isinstance(proof, dict) or proof.get("task_type") != seed["task_type"]:
            raise ValueError("wrong task/schema")
        verify = {"numeral_system": _verify_roman_proof, "gravity": _verify_gravity_proof,
                  "equation_numeric": _verify_equation_proof}[seed["task_type"]]
        verified = bool(verify(seed, proof))
        return {"verified": verified, "proof": proof, "scope": "all fields of a closed structured proof; no free text"}
    except (ValueError, TypeError, KeyError, AttributeError, IndexError, ZeroDivisionError) as error:
        return {"verified": False, "error": repr(error), "scope": "unverified Preview only"}


In [ ]:
SMOKE_TRACE_INSTRUCTIONS = {
    "numeral_system": 'Return one JSON object only with keys task_type="numeral_system", query (integer), components (list of {"arabic": integer, "roman": string}), answer (string). Decompose the query into Roman place-value pieces, then concatenate. Every component must be a canonical Roman numeral and their Arabic values must sum to query.',
    "gravity": 'Return one JSON object only with keys task_type="gravity", g_estimates (list of numbers in example order), g_ols (number), query_t (original time string), query_distance (number before rounding), answer (string). Compute each g_i=2*d_i/t_i^2, then g_ols=2*sum(t_i^2*d_i)/sum(t_i^4), then query_distance=0.5*g_ols*query_t^2. Give intermediate values to at least six significant digits.',
    "equation_numeric": 'Return one JSON object only with keys task_type="equation_numeric", rules (operator to public rule name), example_outputs (list of strings in example order), query_output (string), answer (string). Choose one valid global assignment, recompute every example and the query; keep operator prefixes/suffixes. All non-join rules must share normal/flip family and at most one per semantic group. Public candidates: ' + json.dumps(EQUATION_NUMERIC_PRIOR, ensure_ascii=False),
}


def build_data_designer_smoke(seed_path, endpoint, served_model=MODEL_ID):
    from urllib.parse import urlparse
    from importlib.metadata import version
    if version("data-designer") != "0.10.0": raise ValueError("Use pinned data-designer==0.10.0")
    import data_designer.config as dd
    if urlparse(endpoint).hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise ValueError("Smoke only supports a local vLLM endpoint")
    provider = dd.ModelProvider(name="local-vllm", endpoint=endpoint, provider_type="openai", api_key=None)
    models = [dd.ModelConfig(alias=alias, model=served_model, provider="local-vllm", skip_health_check=True,
        inference_parameters=dd.ChatCompletionInferenceParams(temperature=0.3, top_p=0.9, max_tokens=cap,
            max_parallel_requests=1, timeout=180, extra_body={"chat_template_kwargs": {"enable_thinking": False}, "seed": SEED}))
        for alias, cap in (("local-qwen3-prompt", 1024), ("local-qwen3-trace", 2048))]
    builder = dd.DataDesignerConfigBuilder(model_configs=models)
    builder.with_seed_dataset(dd.LocalFileSeedSource(path=str(seed_path)), sampling_strategy=dd.SamplingStrategy.ORDERED)
    prompt = ('Produce a safe wording variant. Output exactly one of these introductory sentences: ' + json.dumps(SMOKE_PREFIXES) +
              '. Then a newline, then copy the entire original task EXACTLY, including all numbers, operators, formulas, examples, units, and query. No JSON, fences, answer or extra text.\nSEED_ID={{ seed_id }}\nORIGINAL_TASK_BEGIN\n{{ prompt }}\nORIGINAL_TASK_END')
    trace = 'Compute a structured proof. No markdown, no additional keys, no narrative text.\nSEED_ID={{ seed_id }}\n'
    for task, instruction in SMOKE_TRACE_INSTRUCTIONS.items():
        trace += '{% if task_type == "' + task + '" %}' + instruction + '{% endif %}\n'
    trace += 'ORIGINAL_TASK_BEGIN\n{{ prompt }}\nORIGINAL_TASK_END'
    for name, alias, text in (("prompt_variant", "local-qwen3-prompt", prompt), ("trace_variant", "local-qwen3-trace", trace)):
        builder.add_column(dd.LLMTextColumnConfig(name=name, model_alias=alias, prompt=text,
            system_prompt="You follow an exact output contract for a local data generation smoke test. /no_think",
            with_trace=dd.TraceType.LAST_MESSAGE, extract_reasoning_content=True))
    return builder, provider


def configure_smoke_runtime(designer):
    from data_designer.config import RunConfig
    designer.set_run_config(RunConfig(max_in_flight_tasks=1, max_concurrent_row_groups=1,
        max_conversation_restarts=0, max_conversation_correction_steps=0, async_trace=True,
        otel_metrics_port=None))
    return designer


In [ ]:
def screen_data_designer_records(records, seeds, destination, require_transport=False):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if any((destination / name).exists() for name in ("candidates.jsonl", "accepted.jsonl", "summary.json")):
        raise FileExistsError("Do not overwrite a previous NeMo screening result")
    by_id = {row["seed_id"]: row for row in seeds}
    counts = Counter(str(row.get("seed_id", "")) for row in records)
    candidates, accepted, previews = [], [], []
    for seed_id, seed in by_id.items():
        matching = [r for r in records if r.get("seed_id") == seed_id]
        record = matching[0] if len(matching) == 1 else {}
        for kind in ("prompt_variant", "trace_variant"):
            text = record.get(kind)
            reasons = []
            if len(matching) != 1: reasons.append("missing_or_duplicate_seed_output")
            if record.get("prompt") != seed["prompt"] or record.get("task_type") != seed["task_type"]:
                reasons.append("seed_provenance_mismatch")
            try:
                verification = verify_smoke_prompt(seed, text) if kind == "prompt_variant" else verify_smoke_trace(seed, text)
            except Exception as error:
                verification = {"verified": False, "error": repr(error)}
            if not verification["verified"]: reasons.append("variant_verification_failed")
            transport = record.get(kind + "__transport", {})
            if require_transport and (not transport.get("verified_response_match") or transport.get("finish_reason") != "stop"):
                reasons.append("missing_or_truncated_transport")
            # Do not accept reasoning_content that is not covered by the closed JSON proof.
            if record.get(kind + "__reasoning_content"): reasons.append("unverified_reasoning_content")
            candidate = {"candidate_id": seed_id + ":" + kind, "seed_id": seed_id,
                "task_type": seed["task_type"], "source": "nemo_local_vllm", "variant_kind": kind,
                "prompt": text.strip() if kind == "prompt_variant" and isinstance(text, str) else seed["prompt"],
                "answer": seed["answer"], "variant_content": text, "verification": verification, "transport": transport,
                "verifier_version": SMOKE_VERIFIER_VERSION, "eligible_for_sft": False}
            leakage = audit_synthetic_leakage(candidate, real_index)
            reasons.extend(leakage["rejection_reasons"])
            candidate.update(leakage_audit=leakage, leakage_status=leakage["status"],
                verification_status="accepted" if not reasons else "preview", rejection_reasons=reasons)
            candidates.append(candidate)
            (accepted if not reasons else previews).append(candidate)
    _write_jsonl(destination / "candidates.jsonl", candidates)
    _write_jsonl(destination / "accepted.jsonl", accepted)
    _write_jsonl(destination / "preview.jsonl", previews)
    cross = _real_synthetic_cross_audit(candidates, destination)
    status = synthetic_accepted_status(accepted)
    expected = 2 * len(seeds)
    complete = len(records) == len(seeds) and set(counts) == set(by_id) and all(v == 1 for v in counts.values())
    summary = {"status": "passed" if complete and len(accepted) == expected else
                         ("empty_accepted" if not accepted else "completed_with_previews"),
        "transport_complete": complete, "seed_rows": len(seeds), "expected_outputs": expected,
        "received_seed_rows": len(records), "nonempty_outputs": sum(isinstance(c["variant_content"], str) and bool(c["variant_content"].strip()) for c in candidates),
        "accepted": len(accepted), "preview": len(previews), "merge_into_sft": False,
        "by_task": {task: {"accepted": sum(r["task_type"] == task for r in accepted),
                           "preview": sum(r["task_type"] == task for r in previews)} for task in SMOKE_TASKS}}
    manifest = {"schema_version": "nemo-smoke-accepted/v1", "status": status, "accepted_rows": len(accepted),
        "all_accepted_verified_and_audited": bool(accepted) and all(r["verification"]["verified"] and r["leakage_status"] == "passed" for r in accepted),
        "accepted_sha256": hashlib.sha256((destination / "accepted.jsonl").read_bytes()).hexdigest(),
        "verifier_version": SMOKE_VERIFIER_VERSION, "split_sha256": cross["split_sha256"], "merge_into_sft": False}
    (destination / "accepted_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    (destination / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
    return summary


In [ ]:
def run_data_designer_smoke():
    # Explicit GPU phase; preparation and dependency/API validation do not call preview.
    from data_designer.interface import DataDesigner
    from utils.run_synthetic_smoke import verify_prepared_run, local_endpoint_preflight, capture_local_requests, attach_transport, write_new
    destination = synthetic_dir / "data_designer"
    verify_prepared_run(synthetic_dir)
    if (destination / "gpu_attempt.json").exists():
        raise FileExistsError("GPU attempt already exists; use a new SYNTHETIC_RUN_ID")
    endpoint = os.environ.get("LOCAL_VLLM_ENDPOINT", "http://127.0.0.1:18081/v1")
    local_endpoint_preflight(endpoint, MODEL_ID)
    seed_path = destination / "public_seeds.jsonl"
    started = time.perf_counter()
    write_new(destination / "gpu_attempt.json", {"status": "started", "endpoint": endpoint,
        "model": MODEL_ID, "started_at_utc": datetime.now(timezone.utc).isoformat()})
    try:
        with capture_local_requests(endpoint, destination / "http_requests.jsonl") as captured_endpoint:
            builder, provider = build_data_designer_smoke(seed_path, captured_endpoint)
            designer = DataDesigner(artifact_path=destination / "nemo_artifacts", model_providers=[provider])
            configure_smoke_runtime(designer)
            designer.validate(builder)
            preview = designer.preview(config_builder=builder, num_records=9)
            records = preview.dataset.to_dict(orient="records")
        transport = attach_transport(records, destination / "http_requests.jsonl")
        _write_jsonl(destination / "raw_model_outputs.jsonl", records)
        seeds = [json.loads(line) for line in (destination / "private_seeds.jsonl").read_text().splitlines()]
        summary = screen_data_designer_records(records, seeds, destination, require_transport=True)
        summary.update(transport=transport, gpu_generation_requested=True, wall_seconds=time.perf_counter()-started,
                       model=MODEL_ID, endpoint=endpoint, synthetic_run_id=synthetic_run_id)
        (destination / "gpu_result.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
        return summary
    except Exception as error:
        result = {"status": "failed_runtime", "error": repr(error), "accepted": 0,
                  "wall_seconds": time.perf_counter()-started, "verified": False}
        (destination / "gpu_result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
        raise


In [ ]:
def _synthetic_artifact_reference(path):
    path = Path(path)
    return str(path.relative_to(project_root)) if path.is_relative_to(project_root) else str(path)


def write_synthetic_accepted_manifest(accepted, cross_audit):
    accepted_path = synthetic_dir / "synthetic_accepted.jsonl"
    accepted_manifest = {
        "schema_version": "nemotron-synthetic-accepted/v1", "status": synthetic_accepted_status(accepted),
        "accepted_sha256": hashlib.sha256(accepted_path.read_bytes()).hexdigest(), "accepted_rows": len(accepted),
        "audit_sha256": hashlib.sha256((synthetic_dir / "synthetic_real_leakage_audit.json").read_bytes()).hexdigest(),
        "dataset_manifest_sha256": cross_audit["dataset_manifest_sha256"], "split_sha256": cross_audit["split_sha256"],
        "verifier_version": SYNTHETIC_VERIFIER_VERSION,
        "numeral_query_exposure_policy": "report_only",
        "numeral_query_exposure_sha256": {name: hashlib.sha256((synthetic_dir / name).read_bytes()).hexdigest()
                                           for name in ("numeral_query_exposure.json", "numeral_query_exposure.csv")},
        "all_accepted_numeral_exposure_audited": all(row["leakage_audit"]["numeral_query_exposure"]["status"] == "audited"
                                                     for row in accepted if row["task_type"] == "numeral_system"),
        "all_accepted_verified_and_audited": bool(accepted) and all(row.get("verification_status") == "accepted" and row.get("leakage_status") == "passed" for row in accepted),
    }
    (synthetic_dir / "accepted_manifest.json").write_text(json.dumps(accepted_manifest, ensure_ascii=False, indent=2), encoding="utf-8")


### 第二轮：独立过程审查与单因素修复
原有 Generator、Solver、Verifier、三集合 Audit 保留。下面的审查函数只用独立 Verifier 的计算路径，不读取隐藏规则；最终答案仍使用官方评分。过程检查的字符串必须精确匹配，不能用 1% 评分容差掩盖算术错误。
R0 为原配置复测。A 仅调整 Prompt；B 仅对 Trace 使用严格结构化 Schema；C 仅加入 Solver 从公开题面推导的辅助上下文。A/B/C 分别对照 R0，使用原 9 个 Seed、相同模型 Revision、采样参数与 Token 上限。输出仅供筛查，不自动进入 SFT。

In [ ]:
def audit_equation_process(seed, text):
    """Independent Numeric24 proof checks; ambiguous official rows remain scored normally."""
    result = {"format_valid": False, "final_correct": False, "exact_trace_pass": False,
              "process_checks": {"example_fraction": 0., "legal_assignment": False,
                                 "query_calculation": False, "answer_consistency": False}}
    try:
        def unique(pairs):
            obj = {}
            for key, value in pairs:
                if key in obj: raise ValueError("duplicate JSON key")
                obj[key] = value
            return obj
        proof = json.loads(text, object_pairs_hook=unique,
                           parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
        keys = {"task_type", "rules", "example_outputs", "query_output", "answer"}
        if (not isinstance(proof, dict) or set(proof) != keys or proof["task_type"] != "equation_numeric"
            or not isinstance(proof["rules"], dict) or not isinstance(proof["example_outputs"], list)
            or not all(type(v) is str for v in proof["example_outputs"])
            or type(proof["query_output"]) is not str or type(proof["answer"]) is not str):
            return result
        examples, query = _parse_numeric_prompt(seed["prompt"])
        operators = list(dict.fromkeys([e[0][2] for e in examples] + [query[2]]))
        rules = proof["rules"]
        result["format_valid"] = (set(rules) == set(operators)
            and all(type(v) is str and v in NUMERIC24_RULE_BY_NAME for v in rules.values())
            and len(proof["example_outputs"]) == len(examples))
        result["final_correct"] = numeric_or_text_match(proof["answer"], seed["answer"])
        if not result["format_valid"]: return result
        legal = bool(_numeric_verifier_valid_assignments({op: [rules[op]] for op in operators}, operators))
        matches = [_smoke_exact_output(_numeric_verify_output(rules[e[2]], e), claimed)
                   and _smoke_exact_output(claimed, expected)
                   for (e, expected), claimed in zip(examples, proof["example_outputs"])]
        explains_all = legal and bool(matches) and all(matches)
        query_ok = explains_all and _smoke_exact_output(_numeric_verify_output(rules[query[2]], query), proof["query_output"])
        consistency = query_ok and _smoke_exact_output(proof["query_output"], proof["answer"])
        result["process_checks"] = {"example_fraction": sum(matches) / len(examples),
            "legal_assignment": legal, "query_calculation": query_ok, "answer_consistency": consistency}
        result["exact_trace_pass"] = bool(explains_all and query_ok and consistency)
        result["example_checks"] = matches
        result["proof"] = proof
        return result
    except (ValueError, TypeError, KeyError, IndexError, ZeroDivisionError) as error:
        result["error"] = repr(error)
        return result

In [ ]:
def smoke_trace_schema(task):
    """The structured generation schema does not change independent acceptance rules."""
    string = {"type": "string"}; number = {"type": "number"}
    fields = {"task_type": {"type": "string", "enum": [task]}, "answer": string}
    if task == "numeral_system":
        fields.update(query={"type": "integer"}, components={"type": "array", "minItems": 1,
            "items": {"type": "object", "properties": {"arabic": {"type": "integer", "minimum": 1}, "roman": string},
                      "required": ["arabic", "roman"], "additionalProperties": False}})
    elif task == "gravity":
        fields.update(g_estimates={"type": "array", "items": number}, g_ols=number,
                      query_t=string, query_distance=number)
    elif task == "equation_numeric":
        fields.update(rules={"type": "object", "additionalProperties": {"type": "string", "enum": list(NUMERIC24_RULE_BY_NAME)}},
                      example_outputs={"type": "array", "items": string}, query_output=string)
    else: raise ValueError(task)
    return {"type": "object", "properties": fields, "required": list(fields), "additionalProperties": False}


def smoke_solver_context(seed):
    # All inputs come from the public prompt. No hidden rule or standard Query answer.
    if seed["task_type"] == "equation_numeric":
        examples, query = _parse_numeric_prompt(seed["prompt"])
        solved = solve_equation_numeric_prompt(seed["prompt"])
        return {"examples": examples, "query": query, "candidate_rules": solved["candidates_by_operator"]}
    if seed["task_type"] == "gravity":
        examples, query = _gravity_prompt_parts(seed["prompt"])
        return {"examples": [{"t": t, "d": d, "t_squared": float(t)**2} for t, d in examples],
                "query_t": query, "formula": "g_i=2*d/t^2; g_ols=2*sum(t^2*d)/sum(t^4); d_query=.5*g_ols*t_query^2"}
    examples, query = parse_numeral_prompt(seed["prompt"])
    return {"examples": examples, "query": query, "places": [query // 1000 * 1000, query // 100 % 10 * 100,
                                                              query // 10 % 10 * 10, query % 10]}

In [ ]:
def build_round2_data_designer(seed_path, endpoint, experiment="R0", task=None):
    """A/B/C are independent interventions against exactly the same R0 baseline."""
    import data_designer.config as dd
    if experiment not in {"R0", "A", "B", "C"}: raise ValueError(experiment)
    builder, provider = build_data_designer_smoke(seed_path, endpoint)
    config = builder.build()
    columns = list(config.columns)
    models = list(config.model_configs)
    if experiment == "A":
        columns[0] = columns[0].model_copy(update={"prompt":
            'Produce a safe wording variant. Return EXACTLY "Solve the task described below.\n" followed by the original task. '
            'The markers below are input delimiters and must NOT be output. Never solve or add examples. '
            'Copy the full task including its introductory sentences and all newlines.\n'
            'SEED_ID={{ seed_id }}\nORIGINAL_TASK_BEGIN\n{{ prompt }}\nORIGINAL_TASK_END'})
        columns[1] = columns[1].model_copy(update={"prompt": columns[1].prompt +
            '\nSelf-check required keys and types before output. rules MUST be an operator-to-rule-name object, '
            'not a list of formulas. example_outputs MUST cover EVERY example in order. '
            'Include answer. Roman components are positive place values, no zero pieces. '
            'Gravity query_t is the queried time without units, not an example time.'})
    elif experiment == "B":
        if task not in SMOKE_TASKS: raise ValueError("B requires a separate task-homogeneous seed shard")
        source = columns[1].model_dump(exclude={"column_type"})
        schema = smoke_trace_schema(task)
        columns[1] = dd.LLMStructuredColumnConfig(**source, output_format=schema)
        extra = dict(models[1].inference_parameters.extra_body or {})
        extra["response_format"] = {"type": "json_schema", "json_schema": {"name": task + "_trace", "strict": True, "schema": schema}}
        params = models[1].inference_parameters.model_copy(update={"extra_body": extra})
        models[1] = models[1].model_copy(update={"inference_parameters": params})
    elif experiment == "C":
        columns[1] = columns[1].model_copy(update={"prompt": columns[1].prompt +
            '\nSOLVER_CONTEXT_FROM_PUBLIC_PROMPT={{ solver_context }}\nRecompute and verify; context is not a final answer.'})
    rebuilt = dd.DataDesignerConfigBuilder(model_configs=models)
    rebuilt.with_seed_dataset(dd.LocalFileSeedSource(path=str(seed_path)), sampling_strategy=dd.SamplingStrategy.ORDERED)
    for column in columns: rebuilt.add_column(column)
    return rebuilt, provider

In [ ]:
def audit_closed_completion(seed, text):
    """RL diagnostic scope: final closed JSON. Never certify unparsed thinking text."""
    text = str(text)
    thinking_present = "</think>" in text
    proof_text = text.rsplit("</think>", 1)[-1].strip() if thinking_present else text.strip()
    audit = {"format_valid": False, "final_correct": False, "exact_trace_pass": False,
             "thinking_present_unverified": thinking_present, "scope": "closed final JSON only",
             "process_checks": {"example_fraction": 0., "legal_assignment": False,
                                "query_calculation": False, "answer_consistency": False}}
    try:
        def unique(pairs):
            obj = {}
            for key, value in pairs:
                if key in obj: raise ValueError("duplicate key")
                obj[key] = value
            return obj
        proof = json.loads(proof_text, object_pairs_hook=unique,
                           parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
        if not isinstance(proof, dict): return audit
        predicted = proof.get("answer")
        audit["predicted"] = predicted
        # Final scoring remains official; strict process checks are separate.
        audit["final_correct"] = type(predicted) is str and numeric_or_text_match(predicted, seed["answer"])
        if seed["task_type"] == "equation_numeric":
            process = audit_equation_process(seed, proof_text)
            audit.update(process)
            # A valid explicit final answer is scored even when other schema fields fail.
            audit["final_correct"] = type(predicted) is str and numeric_or_text_match(predicted, seed["answer"])
        else:
            import jsonschema
            try:
                jsonschema.validate(proof, smoke_trace_schema(seed["task_type"]))
                audit["format_valid"] = True
            except jsonschema.ValidationError:
                pass
            audit["exact_trace_pass"] = bool(audit["format_valid"] and verify_smoke_trace(seed, proof_text)["verified"])
        return audit
    except (ValueError, TypeError, KeyError) as error:
        audit["error"] = repr(error)
        return audit


In [ ]:
def audit_gravity_formula(text):
    """Symbolic formula diagnostic only; finding a formula does not verify a CoT."""
    import ast
    from fractions import Fraction
    def monomial(node):
        if isinstance(node, ast.Name) and node.id in {"d", "t"}:
            return Fraction(1), int(node.id == "d"), int(node.id == "t")
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return Fraction(str(node.value)), 0, 0
        if isinstance(node, ast.BinOp):
            a, b = monomial(node.left), monomial(node.right)
            if isinstance(node.op, ast.Mult): return a[0]*b[0], a[1]+b[1], a[2]+b[2]
            if isinstance(node.op, ast.Div): return a[0]/b[0], a[1]-b[1], a[2]-b[2]
            if isinstance(node.op, ast.Pow) and b[1:] == (0, 0) and b[0].denominator == 1 and abs(b[0]) <= 4:
                power = int(b[0]);return a[0]**power, a[1]*power, a[2]*power
        raise ValueError("unsupported symbolic formula")
    candidates = []
    normalized = str(text).replace("²", "^2").replace("×", "*").replace("·", "*")
    for match in re.finditer(r"(?<![A-Za-z_])g\s*=\s*([^\n;]+)", normalized):
        expression = match.group(1).strip().rstrip(".,")
        expression = re.sub(r"(?<=\d)(?=[dt])", "*", expression).replace("^", "**")
        try:
            if len(expression) > 128: raise ValueError("formula too long")
            value = monomial(ast.parse(expression, mode="eval").body)
            valid = value == (Fraction(2), 1, -2)
        except (ValueError, SyntaxError, TypeError, ZeroDivisionError, OverflowError):
            valid = False
        candidates.append({"expression": expression, "relation_verified": valid})
    return {"formula_relation_found": any(r["relation_verified"] for r in candidates),
            "candidates": candidates, "substitutions_verified": False,
            "whole_trace_verified": False, "scope": "g=2*d/t^2 relation only; numerical proof still required"}


### 执行 Pilot（显式开启）

只向当前版本和 Run ID 写入新产物，存在候选时拒绝覆盖。默认完全跳过生成；不自动读取旧版 Bit Pilot。
需要复核已有 Accepted 时，另设 `RUN_TRAIN_ARTIFACT_AUDIT=1`，只写复核报告，不修改 Accepted。


In [ ]:
if run_synthetic:
    synthetic_pilot_summary = run_synthetic_pilot()
else:
    synthetic_pilot_summary = {"mode": "not_run", "reason": "RUN_SYNTHETIC=0; Baseline 不依赖合成数据"}
print({key: synthetic_pilot_summary.get(key) for key in ("mode", "generated", "accepted", "rejected", "by_task")})


## 4. Verifier 负样本与泄漏检查

保留原有错答案、错轨迹、不可识别和重复样本检查；额外注入三个集合的重复与来源 ID，
验证精确重复和规范化重复均被拒绝，只有结构相似时能够通过泄漏检查。

In [ ]:
# Negative tests: each malformed candidate must be rejected by an independent Verifier or Screening.
from copy import deepcopy

candidate_path = synthetic_dir / "synthetic_candidates.jsonl"
candidate_rows = [json.loads(line) for line in candidate_path.read_text(encoding="utf-8").splitlines() if line.strip()] if candidate_path.exists() else []
accepted_by_task = {
    task: next((deepcopy(row) for row in candidate_rows if row.get("task_type") == task and row.get("verification_status") == "accepted"), None)
    for task in TARGET_TASKS
}
# Roman 独立负样本无需运行 Pilot，也不依赖历史 Accepted。
roman_fixture = generate_numeral_sample(np.random.default_rng(SEED))
_solve_candidate(roman_fixture)
assert verify_numeral_sample(roman_fixture)["answer_ok"]
negative_results = []


def _negative_verifier_case(name: str, task_type: str, sample: dict[str, object], expected: str = "reject") -> None:
    result = _screen_verification(sample)
    rejected = not (result.get("answer_ok", False) and result.get("trace_ok", False) and result.get("identifiability", {}).get("answer_unique", False))
    negative_results.append({"case": name, "task_type": task_type, "expected": expected, "passed": bool(rejected), "result": result})


for label, changed in (
    ("wrong_answer", {"answer": "IIII"}),
    ("wrong_trace", {"trace": {"query_number": -1, "components": []}}),
    ("bad_example", {"prompt": re.sub(r"-> [A-Z]+", "-> IIII", roman_fixture["prompt"], count=1)}),
    ("missing_query", {"prompt": "12 -> XII"}),
):
    candidate = deepcopy(roman_fixture); candidate.update(changed)
    _negative_verifier_case("numeral_" + label, "numeral_system", candidate)

if accepted_by_task["gravity"]:
    gravity_case = deepcopy(accepted_by_task["gravity"])
    gravity_case["answer"] = "999999"
    _negative_verifier_case("gravity_wrong_answer", "gravity", gravity_case)
    gravity_case = deepcopy(accepted_by_task["gravity"])
    gravity_case["trace"]["query_t"] = "999"
    _negative_verifier_case("gravity_wrong_trace", "gravity", gravity_case)
    gravity_case["prompt"] = "Now, calculate the falling distance for t = 2s."
    _negative_verifier_case("gravity_unidentifiable_prompt", "gravity", gravity_case)
if accepted_by_task["equation_numeric"]:
    equation_case = deepcopy(accepted_by_task["equation_numeric"])
    equation_case["answer"] = "not-a-number"
    _negative_verifier_case("equation_wrong_answer", "equation_numeric", equation_case)
    equation_case = deepcopy(accepted_by_task["equation_numeric"])
    equation_case["trace"]["query_outputs"] = ["not-the-solver-output"]
    _negative_verifier_case("equation_wrong_trace", "equation_numeric", equation_case)
    equation_case["prompt"] = re.sub(r" = [^\n]+", " = invalid", equation_case["prompt"], count=1)
    _negative_verifier_case("equation_invalid_example_output", "equation_numeric", equation_case)
if accepted_by_task.get("bit_manipulation"):
    bit_case = deepcopy(accepted_by_task["bit_manipulation"])
    bit_case["answer"] = "not-bits"
    _negative_verifier_case("bit_wrong_answer", "bit_manipulation", bit_case)
    bit_case = deepcopy(accepted_by_task["bit_manipulation"])
    bit_case["trace"]["expression"] = "AND(x)"
    _negative_verifier_case("bit_invalid_expression", "bit_manipulation", bit_case)
    bit_case["prompt"] = "\n".join(bit_case["prompt"].splitlines()[:2] + [bit_case["prompt"].splitlines()[-1]])
    _negative_verifier_case("bit_unidentifiable_prompt", "bit_manipulation", bit_case)

# Duplicate prompt must be rejected by Screening even when the Verifier accepts it.
if accepted_by_task["gravity"]:
    duplicate = deepcopy(accepted_by_task["gravity"])
    duplicate_record = screen_synthetic_sample(
        duplicate,
        {prompt_hash(duplicate["prompt"])},
        {synthetic_prompt_fingerprint(duplicate["prompt"])},
        {structure_fingerprint(duplicate["prompt"], duplicate["task_type"])},
        audit_synthetic_leakage(duplicate, real_index),
    )
    negative_results.append({"case": "duplicate_prompt", "task_type": "gravity", "expected": "reject", "passed": bool(not duplicate_record["accepted"] and "duplicate_prompt" in duplicate_record["rejection_reasons"]), "result": duplicate_record})

for split_name in ("train", "validation", "holdout_test"):
    row = split_df[(split_df["split"] == split_name) & (split_df["task_type"] == "gravity")].iloc[0]
    exact = {"prompt": row["prompt"], "task_type": row["task_type"]}
    normalized = {**exact, "prompt": "  " + row["prompt"].replace("\n", "   ").swapcase() + "  "}
    lineage = {"prompt": "unrelated prompt", "task_type": row["task_type"], "source_id": row["id"]}
    for name, sample in (("exact", exact), ("normalized", normalized), ("source_id", lineage)):
        result = audit_synthetic_leakage(sample, real_index)
        negative_results.append({"case": f"{split_name}_{name}_leakage", "passed": result["status"] == "rejected", "rejection_reasons": result["rejection_reasons"]})
    similar = {**exact, "prompt": re.sub(r"\d", "7", row["prompt"])}
    result = audit_synthetic_leakage(similar, real_index)
    assert result["structure_similar_only"] and result["status"] == "passed", result

negative_report = {
    "cases": negative_results,
    "passed": int(sum(case["passed"] for case in negative_results)),
    "total": len(negative_results),
    "all_passed": bool(negative_results) and all(case["passed"] for case in negative_results),
}
(synthetic_dir / "verifier_negative_tests.json").write_text(json.dumps(negative_report, ensure_ascii=False, indent=2), encoding="utf-8")
if not negative_report["all_passed"]:
    raise AssertionError(negative_report)
print({key: value for key, value in negative_report.items() if key != "cases"})

### Numeral 映射暴露的边界检查

独立构造三个集合，覆盖双向暴露、相同目标、题内暴露、未暴露及无效候选。


In [ ]:
exposure_prompt = "1 -> I\n14 -> XIV\nNow, write the number 21 in the Wonderland numeral system."
exposure_references = pd.DataFrame([
    {"id": "train-example", "split": "train", "prompt": exposure_prompt, "answer": "XXI"},
    {"id": "train-target", "split": "train", "prompt": "2 -> II\nNow, write the number 14 in the Wonderland numeral system.", "answer": "XIV"},
    {"id": "val-example", "split": "validation", "prompt": "14 -> XIV\nNow, write the number 9 in the Wonderland numeral system.", "answer": "IX"},
    {"id": "test-target", "split": "holdout_test", "prompt": "30 -> XXX\nNow, write the number 10 in the Wonderland numeral system.", "answer": "X"},
]).assign(task_type="numeral_system", source_row=lambda frame: frame.index.astype(str))
exposure_index = build_real_index(exposure_references)
exposure_candidate = deepcopy(roman_fixture)
exposure_candidate["candidate_id"] = "exposure-fixture"
exposure_candidate["prompt"] = "9 -> IX\n10 -> X\nNow, write the number 14 in the Wonderland numeral system."
_solve_candidate(exposure_candidate)
exposure = audit_synthetic_leakage(exposure_candidate, exposure_index)
assert exposure["status"] == "passed"  # 有限映射暴露只报告，样本/来源审计仍通过。
checks = exposure["numeral_query_exposure"]["by_split"]
assert [row["id"] for row in checks["train"]["query_in_real_examples"]] == ["train-example"]
assert [row["id"] for row in checks["train"]["same_query_in_real_targets"]] == ["train-target"]
assert [row["id"] for row in checks["validation"]["query_in_real_examples"]] == ["val-example"]
assert [row["id"] for row in checks["validation"]["real_queries_in_synthetic_examples"]] == ["val-example"]
assert [row["id"] for row in checks["holdout_test"]["real_queries_in_synthetic_examples"]] == ["test-target"]
assert not checks["holdout_test"]["query_in_real_examples"]
assert not exposure["numeral_query_exposure"]["query_already_in_own_examples"]
assert screen_synthetic_sample(exposure_candidate, set(), set(), set(), exposure)["accepted"]
own_query = {**exposure_candidate, "prompt": "14 -> XIV\n" + exposure_candidate["prompt"]}
assert audit_numeral_query_exposure(own_query, exposure_index["numeral_mapping"])["query_already_in_own_examples"]
unseen = {**exposure_candidate, "prompt": "102 -> CII\nNow, write the number 101 in the Wonderland numeral system.", "answer": "CI"}
unseen_audit = audit_numeral_query_exposure(unseen, exposure_index["numeral_mapping"])
assert unseen_audit["status"] == "audited" and not any(any(hits.values()) for hits in unseen_audit["by_split"].values())
for changed in ({"prompt": "invalid"}, {"answer": "IX"}, {"prompt": exposure_candidate["prompt"].replace("9 -> IX", "9 -> XII")}):
    invalid = {**exposure_candidate, **changed}
    result = audit_synthetic_leakage(invalid, exposure_index)
    assert result["status"] == "rejected" and "numeral_exposure_audit_failed" in result["rejection_reasons"]
assert audit_numeral_query_exposure({"task_type": "gravity"}, exposure_index["numeral_mapping"])["status"] == "not_applicable"
print({"numeral_query_exposure_regression": "passed", "checks": ["three_splits", "both_directions", "same_target", "own_example", "unseen", "invalid_rejected", "report_only"]})


### 可选 CPU 诊断：复核现有合成训练产物

本检查不重新生成数据、不改写 Accepted 文件，也不使用模型输出作标准答案。将现有 Accepted 数据逐条重新交给独立 Verifier，记录来源、文件指纹和失败项。通过只说明当前有限规则空间下的答案、轨迹与可识别性检查通过，不能证明模型能学会，也不能解释未经过 SFT 的基座行为。


In [ ]:
def audit_existing_synthetic_artifact() -> dict:
    source = synthetic_dir / "synthetic_accepted.jsonl"
    records = [json.loads(line) for line in source.read_text().splitlines() if line.strip()]
    rows = []
    for record in records:
        try:
            result = _screen_verification(record)
            checks = {"answer_ok": bool(result.get("answer_ok")),
                      "trace_ok": bool(result.get("trace_ok")),
                      "answer_unique": bool(result.get("identifiability", {}).get("answer_unique")),
                      "accepted_source": record.get("verification_status") == "accepted",
                      "train_prior_source": record.get("source") == "deterministic_prior_generator",
                      "leakage_passed": record.get("leakage_status") == "passed"}
            exposure = audit_numeral_query_exposure(record, real_index["numeral_mapping"])
            checks["numeral_exposure_audited"] = exposure["status"] in {"audited", "not_applicable"}
            failed = [key for key, valid in checks.items() if not valid]
            rows.append({"candidate_id": record["candidate_id"], "task_type": record["task_type"],
                         "checks": checks, "numeral_query_exposure": exposure, "status": "passed" if not failed else "failed", "failed_checks": failed})
        except Exception as error:
            rows.append({"candidate_id": record.get("candidate_id"), "task_type": record.get("task_type"),
                         "status": "failed", "error": repr(error)})
    by_task = {task: {"records": sum(row["task_type"] == task for row in rows),
                      "passed": sum(row["task_type"] == task and row["status"] == "passed" for row in rows)}
               for task in sorted(TARGET_TASKS)}
    report = {"status": "passed" if all(row["status"] == "passed" for row in rows) and rows else "failed",
              "model_loaded": False, "records": len(rows), "by_task": by_task,
              "source": str(source.relative_to(project_root)), "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
              "verifier_version": SYNTHETIC_VERIFIER_VERSION, "scope": "existing screened synthetic artifact; finite course prior only", "rows": rows}
    output = synthetic_dir / "synthetic_artifact_reverification.json"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return report


if os.environ.get("RUN_TRAIN_ARTIFACT_AUDIT", "0") == "1":
    synthetic_artifact_audit = audit_existing_synthetic_artifact()
    print({key: value for key, value in synthetic_artifact_audit.items() if key != "rows"})


## 5. 产物与后续使用

当前版本的 `synthetic/<SYNTHETIC_RUN_ID>/` 保存候选、通过、拒绝、筛查、交叉审计及 `accepted_manifest.json`。
Accepted 内容哈希与新版三个真实集合绑定；历史 Pilot、旧划分和实验结果保持原样。
`numeral_query_exposure.csv/json` 分别保存逐候选、逐集合证据及 Candidates / Accepted 两种口径汇总；内容哈希写入 Accepted Manifest。
映射暴露采用与 00 一致的 `report_only` 策略，Accepted 仍可能包含已暴露映射，不代表数字值 OOD 或映射层面无泄漏。
历史 Accepted 缺少这项审计时，不能据其旧 Manifest 宣称已完成新版映射审计。
NeMo Preview 单独保留，不能成为 Ground Truth 或直接并入 Accepted。
本轮不自动迁移 SFT/RL 输入，也不屏蔽任何任务的未来 RL 入口。三类均计划参加后续 SFT，之后再判断 RL 的必要性。


In [ ]:
if run_synthetic:
    accepted_manifest = json.loads((synthetic_dir / "accepted_manifest.json").read_text(encoding="utf-8"))
    assert accepted_manifest["status"] == ("passed" if accepted_manifest["accepted_rows"] else "empty_accepted")
    assert accepted_manifest["all_accepted_verified_and_audited"] == bool(accepted_manifest["accepted_rows"])
    assert accepted_manifest["all_accepted_numeral_exposure_audited"]
    for name, expected in accepted_manifest["numeral_query_exposure_sha256"].items():
        assert hashlib.sha256((synthetic_dir / name).read_bytes()).hexdigest() == expected
    assert accepted_manifest["accepted_rows"] + synthetic_pilot_summary["rejected"] == synthetic_pilot_summary["generated"]
    assert set(synthetic_pilot_summary["by_task"]) == TARGET_TASKS
    print({"accepted": accepted_manifest["accepted_rows"], "dataset_version": dataset_version,
           "baseline_requires_this_notebook": False, "bit_in_default_pilot": False})


## 6. 显式 NeMo GPU Smoke 入口
CPU 准备、真实包 API 预检与启动命令见 `docs/NEMO_DATA_DESIGNER_SMOKE.md`。不要同时开启旧 Preview 和 Smoke 两个标志。

In [ ]:
if run_data_designer_preview:
    assert not run_synthetic
    data_designer_preview_result = _data_designer_preview()
    print(data_designer_preview_result)

if os.environ.get("RUN_DATA_DESIGNER_SMOKE", "0") == "1":
    assert not run_synthetic and not run_data_designer_preview
    data_designer_smoke_result = run_data_designer_smoke()
    print(data_designer_smoke_result)
